# SIRAP Backend Ecuador · RT-DETRv2-S, OCR y OpenVINO

**Archivo:** `sirap_backend_ecuador.ipynb` · **País:** Ecuador · **Integración:** 16 de noviembre de 2026.

Este notebook conserva el flujo ecuatoriano del adjunto `sirap_backend.ipynb`: escenas COCO,
recortes SAVH Guayaquil, revisión humana, normalización de placas, lista CSV, demostración
en video y consultas opcionales SRI/propietario. Integra las secciones **13, 14, 16, 18 y 25**
del notebook USA adjunto, con todas las subsecciones de cámara y las dependencias necesarias.
La numeración completa es continua: **0–9**, con subsecciones consecutivas dentro de cada bloque.

## Contenido y orden de ejecución

| Sección | Contenido |
|---|---|
| 0. Preparación del entorno | 0.1 dependencias; 0.2 repositorio, rutas Ecuador y controles; 0.3 OCR y persistencia opcional. |
| 1. Datos y análisis exploratorio | 1.1 escenas COCO Ecuador; 1.2 auditoría de particiones; 1.3 recortes SAVH. |
| 2. Preprocesamiento | 2.1 normalización ecuatoriana; 2.2 referencia revisada, duplicados y pendientes SAVH. |
| 3. Modelado y decisión del experimento | 3.1 entrenamiento opcional; 3.2 selección de modelo y requisitos de cámara; 3.3 lista CSV; 3.4 regla de decisión. |
| 4. Cargar el IR OpenVINO de Ecuador | Prioriza el candidato ecuatoriano local o publicado; prepara el pipeline con reglas Ecuador. |
| 5. Prueba básica de OpenVINO | Confirma que el detector ejecuta inferencia; no mide exactitud. |
| 6. Sistema completo en video | 6.1 fuente; 6.2 muestreo; 6.3 CSV/SRI; 6.4 propietario; 6.5 comprobaciones locales; 6.6 ejecución integrada. |
| 7. Cámara en vivo y ejecución local | 7.1 puente Cloudflare; 7.2 lector y reconocimiento Ecuador; 7.3 conexión; 7.4 sesión; 7.5 script para el PC de la cámara. |
| 8. Latencia de inferencia en CPU | Calentamiento y mediana/p95/p99/FPS del detector sobre validación Ecuador. |
| 9. Publicación opcional del IR revisado | Prepara y publica únicamente `models/ecuador/detector/rtdetrv2_s/openvino/`. |

| Sección del notebook USA | Sección integrada aquí |
|---|---|
| 13 | 4 |
| 14 | 5 |
| 16; 16.1–16.5 | 7; 7.1–7.5 |
| 18 | 8 |
| 25 | 9 |

## Rutas y archivos

El código se carga desde [SIRAP_USA_ECUADOR en GitHub](https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR).
Todas las rutas de datos, evaluación, resultados y modelos de este perfil están bajo **`ecuador`**:

| Uso | Ruta relativa al repositorio |
|---|---|
| Configuración de país | `config/countries/ecuador.yaml` |
| Escenas COCO del detector | `datasets/ecuador/detection_coco/` |
| Recortes y revisión SAVH | `datasets/ecuador/savh/` |
| Evaluación Ecuador | `evaluation/ecuador/` |
| Métricas agregadas y calibración | `results/ecuador/metrics/` |
| IR publicable del detector | `models/ecuador/detector/rtdetrv2_s/openvino/` |
| Modelos OCR propios, si se generan aparte | `models/ecuador/ocr/` |
| Ubicación sugerida del notebook | `notebooks/ecuador/sirap_backend_ecuador.ipynb` |

Los checkpoints de entrenamiento, CSV con placas/propietarios y sesiones de cámara se procesan
en `/content/SIRAP_workspace/ecuador/`. La copia a Drive es **opcional**, mediante
`PERSIST_OUTPUTS_TO_DRIVE=True`; usa `SIRAP_USA_ECUADOR/private/ecuador/` para las sesiones
de cámara. Los CSV no se publican automáticamente. El OCR global conserva el nombre técnico
`cct-s-v2-global-model`; las placas se interpretan como Ecuador, sin estado/jurisdicción USA.

**Datos y alcance.** SAVH aporta recortes y transcripciones para revisar el OCR; el detector
necesita escenas completas con todas las placas delimitadas y negativos reales. Copiar un
archivo USA a una carpeta llamada `ecuador` no cambia el país de sus imágenes. Usa escenas
ecuatorianas verificadas para entrenar y evaluar este perfil. No se importan resultados USA
como evidencia de rendimiento Ecuador, ni se entrena con respuestas SRI o nombres de propietario.

**Ejecución.** Ejecuta las celdas en orden. El entrenamiento, la carga del modelo, el video,
las conexiones de cámara, la latencia, las consultas externas y las publicaciones empiezan
desactivados. Para entrenar: prepara COCO en 1, activa `RUN_TRAIN_ECUADOR` y ejecuta 3.1.
Para probar el IR: activa `RUN_INFERENCE`, revisa la selección 3.2 y ejecuta 4–5. Para video:
ejecuta también 6 con un archivo propio y `RUN_VIDEO=True`. Los umbrales 0,70 de detección
y 0,94 de OCR se conservan para la demostración; no se presentan como calibración Ecuador.

**Cámara.** 7.3 comprueba la conexión sin cargar el detector. Las sesiones de reconocimiento
7.4 y el script 7.5 requieren un IR Ecuador y un informe de calibración Ecuador marcado
`usable_for_inference=true`, con el umbral y los hashes del XML/BIN seleccionado (formato en 3.2).
La calibración de la sección 15 USA no fue solicitada ni se ejecuta aquí. No se fabrica ese
informe ni se reutiliza el de otro país. La cámara guarda lecturas para revisión; la regla
CSV de 3.4 se usa en la demostración en video. Ningún flujo de este notebook acciona una barrera.

**Publicación.** 9 necesita `GITHUB_TOKEN` en Secrets de Colab con permiso Contents de escritura.
Al activarla se publica el bundle IR; la automatización existente puede sincronizarlo al Space.
La activación de pesos en la configuración de la aplicación es una decisión aparte. Preparar
este archivo no ejecuta entrenamiento, pruebas de cámara ni publicaciones.


## 0. Preparación del entorno
### 0.1 Instalar dependencias

En Colab, selecciona GPU para entrenar. Se conserva el PyTorch con CUDA del runtime.
El detector usa OpenVINO nativo. El OCR usa su artefacto ONNX: en CPU Intel se instala
ONNX Runtime con proveedor OpenVINO; en otros procesadores, el proveedor CPU.
Si Colab solicita reiniciar, reinicia antes de continuar con 0.2.


In [ ]:
import platform
import re
import subprocess
import sys
from pathlib import Path

def detect_intel_cpu():
    """Use the OpenVINO EP package only when the host reports an Intel CPU."""
    details = f"{platform.processor()} {platform.uname().processor}"
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.is_file():
        try:
            details += " " + cpuinfo.read_text(errors="ignore")
        except OSError:
            pass
    has_intel = bool(re.search(r"genuineintel|intel\(r\)|\bintel\b", details, re.I))
    has_amd = bool(re.search(r"authenticamd|\bamd\b", details, re.I))
    return has_intel and not has_amd

INTEL_CPU_DETECTED = detect_intel_cpu()
OCR_PACKAGE = (
    "fast-plate-ocr[onnx-openvino]==1.1.0"
    if INTEL_CPU_DETECTED
    else "fast-plate-ocr[onnx]==1.1.0"
)
packages = [
    "transformers>=5.4,<6",
    "accelerate>=1.2,<2",
    "pycocotools>=2.0.8",
    "openvino>=2025.4,<2027",
    OCR_PACKAGE,
    "opencv-python-headless==4.12.0.88",
    "matplotlib>=3.8",
    "pandas>=2.2,<4",
    "PyYAML>=6,<7",
    "requests>=2.31,<3",
    "tqdm>=4.66,<5",
    "gdown>=5,<6",
    "python-Levenshtein>=0.27",
    "pillow",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("OCR backend package:", OCR_PACKAGE)
print("Intel CPU detected:", INTEL_CPU_DETECTED)
print("Dependencies installed. If Colab asks to restart the runtime, restart it and then continue from the repository setup section.")


### 0.2 Repositorio, rutas de Ecuador y controles de ejecución

Los insumos se leen del clon GitHub bajo `datasets/ecuador`. Los candidatos de entrenamiento
se guardan aparte en `WORK_DIR`; la publicación de 9 coloca solo el IR en `models/ecuador`.
El repositorio no se actualiza ni se publica automáticamente. Si reutilizas un clon,
comprueba que contiene el código actual. Los CSV de la sesión permanecen en el espacio local.
Activa Drive únicamente si quieres copiar los scripts y sesiones de cámara terminadas.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, csv, hashlib, zipfile, re, math, inspect
import shutil, time, platform
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import Levenshtein
import yaml
from IPython.display import display

REPOSITORY_URL = 'https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git'
REPO_ROOT = Path(os.environ.get('SIRAP_REPO_ROOT', '/content/SIRAP_USA_ECUADOR')).expanduser().resolve()
if not (REPO_ROOT / '.git').is_dir():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f'La carpeta no es un clon Git vacío: {REPO_ROOT}. Configura SIRAP_REPO_ROOT.')
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

COUNTRY = 'ecuador'
COUNTRY_CONFIG_PATH = REPO_ROOT / 'config/countries/ecuador.yaml'
COUNTRY_CONFIG = yaml.safe_load(COUNTRY_CONFIG_PATH.read_text(encoding='utf-8'))
DATASET_DIR = REPO_ROOT / 'datasets/ecuador'
EVALUATION_DIR = REPO_ROOT / 'evaluation/ecuador'
RESULTS_DIR = REPO_ROOT / 'results/ecuador'
METRICS_DIR = RESULTS_DIR / 'metrics'
REPO_MODEL_OUTPUT = REPO_ROOT / 'models/ecuador/detector/rtdetrv2_s'
OCR_MODEL_OUTPUT = REPO_ROOT / 'models/ecuador/ocr'
WORK_DIR = Path(os.environ.get('SIRAP_ECUADOR_WORK_DIR', '/content/SIRAP_workspace/ecuador')).expanduser().resolve()
OCR_RESULTS_DIR = WORK_DIR / 'ocr'
for directory in (WORK_DIR, OCR_RESULTS_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SAVH_ZIP_CANDIDATES = [DATASET_DIR / 'savh/revision_visual/savh_ground_truth_subset.zip',
                       WORK_DIR / 'savh_ground_truth_subset.zip']
REVIEW_CSV_FALLBACK = DATASET_DIR / 'savh/ocr/revision_placas_142_leida.csv'
USE_GOOGLE_SHEET = False
REVIEW_SHEET_ID = ''  # La revisión versionada funciona sin autenticar Google.
PLATE_PATTERNS = tuple(tuple(v) for v in COUNTRY_CONFIG.get('placas', {}).get('patrones', ((3, 4), (3, 3))))
AUTHORIZED_LIST_PATH = WORK_DIR / 'authorized_plates.csv'
MODEL_ID = 'merve/license-plates-rtdetrv2'
IMAGE_SIZE = 640
DETECTOR_THRESHOLD = 0.70
OCR_THRESHOLD = 0.94

# Activar solo las etapas que se quieran ejecutar y volver a correr sus celdas.
RUN_TRAIN_ECUADOR = False
RUN_INFERENCE = False
RUN_VIDEO = False
USE_ECUADOR_TRAINED_MODEL = True
USE_BASE_MODEL_FOR_SMOKE = False  # True permite exportar el modelo inicial solo como prueba técnica.
PERSIST_OUTPUTS_TO_DRIVE = False
PUBLISH_SMALL_METRICS_TO_GITHUB = False

DRIVE_ROOT = Path('/content/drive_sirap/MyDrive/SIRAP_USA_ECUADOR')
DRIVE_SCRIPTS_DIR = DRIVE_ROOT / 'generated_scripts/ecuador'
DRIVE_PRIVATE_CAMERA_DIR = DRIVE_ROOT / 'private/ecuador/camera_sessions'
if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive_sirap')

print('País:', COUNTRY, '| trabajo:', WORK_DIR)
print('COCO GitHub:', DATASET_DIR / 'detection_coco')
print('Configuración:', COUNTRY_CONFIG_PATH)
print('Demostración: detección', DETECTOR_THRESHOLD, '| OCR', OCR_THRESHOLD)


### 0.3 OCR y persistencia opcional

Se definen el lector OCR común, la copia opcional a Drive y la publicación explícita de
métricas agregadas. Definirlos no descarga pesos OCR, no abre una cámara ni escribe en GitHub.
Los CSV de placas/propietarios no se pasan a la función de publicación de métricas.


In [ ]:
def persist_file_to_drive(source_path, drive_path):
    """Copia opcional; sin Drive activado devuelve el archivo local."""
    source_path, drive_path = Path(source_path), Path(drive_path)
    if not source_path.is_file():
        raise FileNotFoundError(f'Falta el archivo local: {source_path}')
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return source_path
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    if source_path.resolve() != drive_path.resolve():
        shutil.copy2(source_path, drive_path)
    return drive_path


def persist_tree_to_drive(source_dir, drive_dir):
    """Conserva el destino existente y añade los resultados terminados."""
    source_dir, drive_dir = Path(source_dir), Path(drive_dir)
    if not source_dir.is_dir():
        raise FileNotFoundError(f'Falta la carpeta local: {source_dir}')
    if not PERSIST_OUTPUTS_TO_DRIVE or source_dir.resolve() == drive_dir.resolve():
        return source_dir
    drive_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir, drive_dir, dirs_exist_ok=True)
    return drive_dir


def publish_small_metrics(paths, message):
    """Publica únicamente archivos agregados explícitos bajo results/ecuador/metrics."""
    if not PUBLISH_SMALL_METRICS_TO_GITHUB:
        return []
    relative = []
    for path in map(Path, paths):
        path = path.resolve()
        path.relative_to(METRICS_DIR.resolve())
        if path.is_file():
            relative.append(path.relative_to(REPO_ROOT.resolve()).as_posix())
    staged = subprocess.run(['git', '-C', str(REPO_ROOT), 'diff', '--cached', '--name-only'],
                            check=True, capture_output=True, text=True).stdout.strip()
    if staged:
        raise RuntimeError('Hay archivos previamente preparados en Git. Revísalos antes de publicar métricas.')
    from scripts.notebook_io import publish_artifacts
    return publish_artifacts(REPO_ROOT, relative, message)


# The detector uses native OpenVINO IR. OCR keeps the ONNX artifact and selects
# OpenVINO Execution Provider on Intel CPUs, or ONNX Runtime CPU elsewhere.
import platform
import re
from types import SimpleNamespace

import cv2
import onnxruntime as ort
import requests
import yaml
from fast_plate_ocr import LicensePlateRecognizer

OCR_MODEL_NAME = "cct-s-v2-global-model"

def detect_intel_cpu():
    """Detect Intel from CPU-identification strings, not from the GPU or OS name."""
    details = f"{platform.processor()} {platform.uname().processor}"
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.is_file():
        try:
            details += " " + cpuinfo.read_text(errors="ignore")
        except OSError:
            pass
    has_intel = bool(re.search(r"genuineintel|intel\(r\)|\bintel\b", details, re.I))
    has_amd = bool(re.search(r"authenticamd|\bamd\b", details, re.I))
    return has_intel and not has_amd

INTEL_CPU_DETECTED = detect_intel_cpu()

def create_serial_ocr_model(model_name=OCR_MODEL_NAME):
    """Create the ORT OCR session with the provider required for this CPU vendor."""
    available = ort.get_available_providers()
    if INTEL_CPU_DETECTED:
        if "OpenVINOExecutionProvider" not in available:
            raise RuntimeError(
                "An Intel CPU was detected, but OpenVINOExecutionProvider is unavailable. "
                "Rerun the dependency cell so it installs fast-plate-ocr[onnx-openvino]."
            )
        providers = ["OpenVINOExecutionProvider"]
        if "CPUExecutionProvider" in available:
            providers.append("CPUExecutionProvider")
    else:
        if "CPUExecutionProvider" not in available:
            raise RuntimeError(
                "The non-Intel OCR route requires ONNX Runtime's CPUExecutionProvider."
            )
        providers = ["CPUExecutionProvider"]

    model = LicensePlateRecognizer(model_name, providers=providers)
    active = model.model.get_providers()
    if INTEL_CPU_DETECTED and (
        not active or active[0] != "OpenVINOExecutionProvider"
    ):
        raise RuntimeError(
            "The Intel OCR session did not register OpenVINOExecutionProvider first. "
            f"Registered providers: {active}"
        )
    if not INTEL_CPU_DETECTED and (
        not active or active[0] != "CPUExecutionProvider"
    ):
        raise RuntimeError(
            f"The non-Intel OCR session must use ONNX Runtime CPU; registered: {active}"
        )
    return model, active

class RuntimeSelectedSerialReader:
    """ONNX OCR reader with an Intel OpenVINO EP / non-Intel ORT CPU selection."""

    def __init__(self, model_name=OCR_MODEL_NAME):
        self.model, self.providers = create_serial_ocr_model(model_name)
        self.backend = (
            "ONNX Runtime + OpenVINOExecutionProvider"
            if INTEL_CPU_DETECTED
            else "ONNX Runtime + CPUExecutionProvider"
        )
        self.runtime_name = self.backend

    def run(self, *args, **kwargs):
        return self.model.run(*args, **kwargs)

    def read(self, crop):
        from src.data_processing import clean_text

        prediction = self.run(
            np.ascontiguousarray(crop),
            return_confidence=True,
        )[0]
        text = clean_text(prediction.plate)
        probabilities = np.asarray(
            prediction.char_probs,
            dtype=np.float32,
        ).ravel()[:max(len(text), 1)]
        if not probabilities.size:
            return text, 0.0, 0.0
        return text, float(probabilities.min()), float(probabilities.mean())


## 1. Datos y análisis exploratorio
### 1.1 Escenas ecuatorianas para el detector

Coloca tu exportación COCO en `ECUADOR_COCO_DIR`, con `train/` y `valid/` (o `val/`)
y su `_annotations.coco.json`. Reserva `test/` para la evaluación posterior. Las cajas
incluyen placas pequeñas, del fondo y parcialmente ocultas; una placa ilegible sigue
siendo una placa. Las escenas negativas llevan lista de anotaciones vacía.

Separa vehículos, clips y sesiones entre particiones **antes** del entrenamiento.
La comprobación siguiente detecta archivos idénticos entre particiones y, cuando están
presentes, IDs `vehicle_id`, `clip_id` o `session_id` compartidos. Sin esos IDs no puede
demostrar independencia de vehículos o escenas. Los datos del descargador USA no se
presentan como datos ecuatorianos.

Ruta por defecto: `datasets/ecuador/detection_coco/` del clon GitHub.
Las carpetas con solo `.gitkeep` no constituyen un dataset COCO listo.


In [ ]:
ECUADOR_COCO_DIR = DATASET_DIR / 'detection_coco'
MODEL_OUTPUT_DIR = WORK_DIR / 'models/detector/rtdetrv2_s/seed_2'
MODEL_OUTPUT = MODEL_OUTPUT_DIR
TRAIN_RUN_DIR = WORK_DIR / 'runs/rtdetrv2_s/seed_2'
TRAINING_MODEL_SOURCE='merve/license-plates-rtdetrv2'
# Para continuar exactamente desde pesos USA se necesita el checkpoint Transformers
# (config.json + model.safetensors), no el IR OpenVINO. Asigna su carpeta aquí si existe.
EPOCHS=30
PATIENCE=8
BATCH_SIZE=8
TRAIN_SEED=2


def validate_ecuador_coco(root):
    from src.train import _annotation_file, CocoPlateDataset
    digests,groups,summary={},{},[]
    for split in ('train','val','test'):
        try: path=_annotation_file(root,split)
        except FileNotFoundError:
            if split=='test': continue
            raise
        payload=json.loads(path.read_text())
        images=payload.get('images',[]);annotations=payload.get('annotations',[])
        if not images: raise ValueError(f'Sin imágenes en {split}')
        by_id={row['id']:row for row in images}
        if len(by_id)!=len(images): raise ValueError(f'IDs de imagen repetidos en {split}')
        dataset=CocoPlateDataset(root,split,None)
        for row in images:
            image_path=dataset._image_path(row['file_name'])
            with Image.open(image_path) as image:
                if image.size!=(int(row['width']),int(row['height'])):
                    raise ValueError(f'Dimensiones COCO incorrectas en {split}: {image_path.name}')
            digest=hashlib.sha256(image_path.read_bytes()).hexdigest()
            if digest in digests and digests[digest]!=split:
                raise ValueError(f'Imagen idéntica compartida entre {digests[digest]} y {split}')
            digests[digest]=split
            for key in ('vehicle_id','clip_id','session_id'):
                if row.get(key) is not None:
                    group=(key,str(row[key]))
                    if group in groups and groups[group]!=split:
                        raise ValueError(f'Grupo {key} compartido entre {groups[group]} y {split}')
                    groups[group]=split
        for ann in annotations:
            image=by_id.get(ann['image_id'])
            if image is None:raise ValueError(f'Caja sin imagen en {split}')
            x,y,w,h=map(float,ann['bbox'])
            if not all(math.isfinite(v) for v in (x,y,w,h)) or min(x,y)<0 or min(w,h)<=0 or x+w>image['width']+1 or y+h>image['height']+1:
                raise ValueError(f'Caja inválida en {split}: {ann.get("id")}')
        if split in ('train','val') and not annotations:
            raise ValueError(f'{split} necesita placas anotadas, además de escenas negativas')
        summary.append({'split':split,'images':len(images),'plates':len(annotations)})
    return pd.DataFrame(summary)


### 1.2 Revisión de particiones

La auditoría de particiones se ejecuta cuando existen los datos. No ejecuta inferencia
sobre el test ni lo utiliza para elegir épocas, pesos o umbrales.


In [ ]:
from src.train import _annotation_file, CocoPlateDataset

DATA_READY = False
SPLITS = {}
try:
    for split in ('train', 'val'):
        _annotation_file(ECUADOR_COCO_DIR, split)
except FileNotFoundError as error:
    print('COCO Ecuador pendiente:', error)
    print('Completa train/valid con escenas ecuatorianas y sus anotaciones antes de entrenar o medir latencia.')
else:
    display(validate_ecuador_coco(ECUADOR_COCO_DIR))
    SPLITS = {split: {'dataset': CocoPlateDataset(ECUADOR_COCO_DIR, split, None)} for split in ('train', 'val')}
    DATA_READY = True
print('DATA_READY:', DATA_READY)


### 1.3 Datos de lectura: SAVH Guayaquil

`datasets/ecuador/savh/revision_visual/savh_ground_truth_subset.zip`, versionado en GitHub, tiene 739 recortes de placa de SAVH Guayaquil: 472 en `placa_legible/` y 267 en `placa_no_legible/`, copiados sin cambios de las 2000 imágenes del dataset público (capturas del 7, 8 y 11 de junio de 2026). `ground_truth.csv` toma la placa del nombre de cada archivo.

**Esa placa no es una transcripción manual: es la lectura automática de la cámara Dahua.** Lo muestran sus errores: 142 de las 472 placas legibles (30 %) no siguen el formato ecuatoriano, y en la imagen la placa se lee bien (`GSO-7335` aparece como `GS07335`, `GTN-8737` como `6TN8737`, `PBT-2437` como `JPBT2437`). Además, 73 imágenes son copias idénticas de otras (43 grupos; quedan 666 únicas, 399 de ellas legibles) y las 267 no legibles también traen texto, que no se puede comprobar mirando la imagen.

En esta integración SAVH se usa para extraer los recortes, aplicar la normalización
(2.1) y armar la referencia corregida con duplicados y revisión humana (2.2).
Las imágenes pendientes se muestran en grupos de **máximo ocho**, controlados por
`SAVH_SAMPLE_SIZE`. Solo las transcripciones confirmadas o corregidas por una persona
se exportan como etiquetas revisadas. El afinamiento OCR se aborda como un experimento aparte; este notebook prepara
sus referencias y el entrenamiento
del detector RT-DETR con escenas COCO distintas. No calcula EMR/CER ni usa etiquetas
sugeridas como si estuvieran confirmadas.

Si falta el ZIP, se informa y se omite SAVH sin impedir preparar los demás bloques.


Los recuentos anteriores describen la revisión histórica del adjunto. La ejecución
actual muestra los recuentos de los archivos realmente cargados. Las etiquetas sugeridas
o de cámara no se convierten en etiquetas humanas confirmadas. Esta sección prepara OCR;
no entrena el detector con recortes de placa.


In [ ]:
# 1.3 · Extrae los recortes de SAVH Guayaquil (una vez por sesión, en el disco local de Colab)
_missing = [name for name in ("WORK_DIR", "OCR_RESULTS_DIR", "SAVH_ZIP_CANDIDATES") if name not in globals()]
if _missing:
    raise RuntimeError("Faltan variables de celdas anteriores: " + ", ".join(_missing) + ". En una sesión nueva, "
                       "corre primero la sección 0 completa.")

SAVH_DIR = WORK_DIR / "savh_ground_truth_subset"  # disco local de Colab: se extrae en cada sesión
SAVH_SAMPLE_SIZE = 8  # máximo de imágenes visibles de la revisión pendiente

savh_zip = next((p for p in SAVH_ZIP_CANDIDATES if p.exists()), None)
if not (SAVH_DIR / "ground_truth.csv").exists() and savh_zip:
    with zipfile.ZipFile(savh_zip) as archive:
        SAVH_DIR.mkdir(parents=True,exist_ok=True)
        for member in archive.infolist():
            destination=(SAVH_DIR/member.filename).resolve()
            if not destination.is_relative_to(SAVH_DIR.resolve()):
                raise ValueError('El ZIP contiene una ruta fuera de la carpeta SAVH')
        archive.extractall(SAVH_DIR)

savh_gt = None
if (SAVH_DIR / "ground_truth.csv").exists():
    savh_gt = pd.read_csv(SAVH_DIR / "ground_truth.csv", dtype=str, keep_default_na=False)
    savh_gt.insert(0, "fila_gt", range(1, len(savh_gt) + 1))  # fila de datos: la misma numeración de la hoja de revisión
    print(f"{len(savh_gt)} recortes en {SAVH_DIR}")
    display(savh_gt["folder"].value_counts())
else:
    print("Aviso: no se encontró savh_ground_truth_subset.zip en el repositorio ni en /content; se salta SAVH.")


## 2. Preprocesamiento

**Imágenes del detector.** Se usan escenas COCO y el `AutoImageProcessor` del checkpoint RT-DETRv2-S; no se aplican aumentos de Ultralytics. En inferencia se usa OpenVINO y el manifiesto de preprocesamiento del modelo. Antes del OCR se recorta cada caja con margen proporcional de 0,06. Las cajas y las transcripciones se tratan como referencias distintas.

**Texto de las placas.** Esta sección prepara lo que se compara con el OCR: la normalización de cualquier lectura al formato ecuatoriano (2.1) y la referencia corregida de SAVH (2.2).

### 2.1 Normalización de placas
El texto de un lector de placas no siempre es una placa válida. Antes de decidir el acceso hace falta encontrar la placa en el texto, corregir confusiones típicas por posición y validar el formato. fast-plate-ocr ya devuelve solo letras y dígitos, casi siempre con forma de placa: en el análisis histórico del adjunto, 398 de sus 399 lecturas de legibles ya tenían un formato válido y la normalización corrigió una sola (`GP1226` → `GPI226`). La misma normalización se aplica también a la lectura de la cámara de SAVH, que sí trae errores de formato, y a lo que escribe a mano el operador en la app.

Las placas ecuatorianas tienen **3 letras + 3 o 4 dígitos** ([El Universo](https://www.eluniverso.com/noticias/2020/06/26/nota/7885368/placas-carros-como-identificarlas/)). En la referencia de SAVH aparecen también lecturas con 2 letras + 4 o 5 dígitos, pero salen de errores de la cámara (`GB01825` es `GBO-1825`), así que solo se aceptan los dos formatos reales.

`normalize_plate` hace tres pasos:

1. **Busca la placa dentro del texto.** Corta el texto en tramos (por espacios, guiones, saltos de línea y otra puntuación), descarta la leyenda `ECUADOR` impresa sobre la placa si viene en el texto, y prueba cada tramo y las uniones de 2 o 3 tramos seguidos (`GBA 1234`).
2. **Corrige por posición.** En las 3 primeras posiciones (letras) cambia un dígito por la letra parecida (0→O, 1→I, 5→S, 6→G, 8→B) y en las siguientes (dígitos) una letra por el dígito parecido (O→0, D→0, Q→0, I→1, Z→2, B→8); `|` se lee como 1 y `$` como S. Gana el candidato que valida con menos cambios, y se aceptan como máximo 2: en SAVH, ninguna lectura que necesitó 3 era correcta.
3. **Valida el formato.** Si ningún candidato valida, la lectura queda inválida y la regla de decisión la manda a verificación manual.

**Formato para decidir, guardar y mostrar.** `format_plate` escribe cada placa válida como en Ecuador: 3 letras, guion y 4 dígitos (`GBA-1234`). A las placas antiguas de 3 dígitos les agrega un 0 al inicio de los números (`GMS-429` → `GMS-0429`), para que todas queden en el formato XXX-0000; por eso `GMS-429` y `GMS-0429` cuentan como la misma placa. Ese es el formato de la lista de autorizados, de la regla de decisión, de los CSV y del video. La evaluación del OCR con SAVH compara las placas sin guion, porque mide la lectura y no el formato.

La celda termina con casos de prueba (lecturas reales de la cámara de SAVH y textos con la leyenda, separadores y símbolos) que deben pasar antes de seguir.


In [ ]:
# 2.1 · Normalización de placas al formato ecuatoriano
CONFUSIONS_TO_LETTER = {"0": "O", "1": "I", "5": "S", "6": "G", "8": "B"}  # dígito leído donde va una letra
CONFUSIONS_TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "Z": "2", "B": "8"}  # letra leída donde va un dígito
SYMBOLS_TO_CHAR = {"|": "1", "$": "S"}  # símbolos que un lector de texto libre puede escribir en lugar de un carácter
SEPARATORS = " \t\r\n-‐‑‒–—_.·•:,;+\"'`*()[]{}<>"  # cortan el texto; cualquier otro símbolo invalida su tramo
MAX_CORRECTIONS = 2       # en SAVH, ninguna lectura que necesitó 3 cambios era correcta
PLATE_HEADER = "ECUADOR"  # leyenda impresa sobre la placa: se descarta si un lector la transcribe junto con la placa
_SEGMENT = re.compile("[^" + re.escape(SEPARATORS) + "]+")  # tramo: caracteres seguidos sin separadores


def clean_plate_text(text) -> str:
    """Mayúsculas y solo letras y dígitos: 'gba-1234' → 'GBA1234'."""
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


def is_plate_header(segment: str) -> bool:
    """'ECUADOR' y sus lecturas aproximadas ('EQUADOR', 'ECUNDERG'): solo letras y a 3 cambios o menos."""
    return (len(segment) >= 5 and segment.isascii() and segment.isalpha()
            and Levenshtein.distance(segment, PLATE_HEADER) <= 3)


def as_plate(candidate: str, corrections: bool = True):
    """(placa, cambios) si el candidato tiene formato de placa, corrigiendo por posición si hace falta; si no, None."""
    for n_letters, n_digits in PLATE_PATTERNS:
        if len(candidate) != n_letters + n_digits:
            continue
        letters, digits = candidate[:n_letters], candidate[n_letters:]
        if corrections:
            letters = "".join(CONFUSIONS_TO_LETTER.get(c, c) for c in letters)
            digits = "".join(CONFUSIONS_TO_DIGIT.get(c, c) for c in digits)
        plate = letters + digits
        changes = sum(a != b for a, b in zip(plate, candidate))
        if changes <= MAX_CORRECTIONS and re.fullmatch(f"[A-Z]{{{n_letters}}}[0-9]{{{n_digits}}}", plate):
            return plate, changes
    return None


def find_plate(raw_text, search: bool = True, corrections: bool = True) -> tuple:
    """Busca la placa en el texto del OCR → (placa, formato válido, cambios, (inicio, fin) en el texto).

    Prueba cada tramo entre separadores y las uniones de 2 o 3 tramos seguidos ('GBA 1234'), sin la leyenda
    ECUADOR. Gana el candidato que valida con menos cambios y, a igualdad, el que aparece primero. Si ninguno
    valida, devuelve el texto limpio sin la leyenda, marcado como inválido. `search=False` y `corrections=False`
    apagan cada paso, para la ablación."""
    text = str(raw_text).upper()
    if not search:  # el texto entero como un solo candidato
        found = as_plate(clean_plate_text(text), corrections)
        return (found[0], True, found[1], None) if found else (clean_plate_text(text), False, 0, None)
    segments = [(m.start(), m.end(), "".join(SYMBOLS_TO_CHAR.get(c, c) for c in m.group()))
                for m in _SEGMENT.finditer(text)]
    segments = [s for s in segments if not is_plate_header(s[2])]
    best = None
    for i in range(len(segments)):
        for j in range(i + 1, min(i + 3, len(segments)) + 1):
            found = as_plate("".join(s[2] for s in segments[i:j]), corrections)
            if found and (best is None or found[1] < best[2]):
                best = (found[0], True, found[1], (segments[i][0], segments[j - 1][1]))
    return best or (clean_plate_text("".join(s[2] for s in segments)), False, 0, None)


def normalize_plate(raw_text: str) -> tuple[str, bool]:
    """Texto del OCR → (placa normalizada, formato válido)."""
    plate, is_valid, _, _ = find_plate(raw_text)
    return plate, is_valid


def format_plate(text) -> str:
    """Formato de Ecuador para decidir, guardar y mostrar: 3 letras, guion y 4 dígitos ('gba1234' → 'GBA-1234'). A las
    placas antiguas de 3 dígitos se les agrega un 0 al inicio de los números ('GMS-429' → 'GMS-0429'). Un texto que no
    es una placa válida queda limpio (mayúsculas, solo letras y dígitos), sin guion."""
    plate = clean_plate_text(text)
    match = re.fullmatch(r"([A-Z]{3})(\d{3,4})", plate)
    return f"{match.group(1)}-{match.group(2).zfill(4)}" if match else plate


def plate_confidence(text: str, char_probs) -> float:
    """Confianza del OCR en la placa: la menor probabilidad entre sus letras y dígitos dentro del texto
    (sin placa válida, entre todas las letras y dígitos). NaN si no hay probabilidades."""
    if char_probs is None or len(char_probs) != len(text):
        return float("nan")
    span = find_plate(text)[3]
    start, end = span if span and len(text.upper()) == len(text) else (0, len(text))
    probs = [p for c, p in zip(text[start:end], char_probs[start:end]) if c.isalnum() or c in SYMBOLS_TO_CHAR]
    return float(min(probs)) if probs else 0.0


# Casos de prueba: lecturas reales de la cámara en SAVH (revisión de placas) y textos con la leyenda ECUADOR,
# separadores y símbolos. Cada uno debe dar la placa esperada.
for raw, expected in [("GB01825", "GBO1825"), ("GS11034", "GSI1034"), ("6TK6566", "GTK6566"), ("GTE5D45", "GTE5045"),
                      ("GP1986", "GPI986"), ("gba-1234", "GBA1234"), ("ECUADOR\nGMS-429", "GMS429"),
                      ("EQUIDOR\nGTE-7040", "GTE7040"), ("ECUNDERG\nGB0-6445", "GBO6445"), ("G5V-3233", "GSV3233"),
                      ("G50-6164", "GSO6164"), ("G|G-430", "GIG430"), ("GSH+2315", "GSH2315"), ("GTT·4568", "GTT4568"),
                      ("ECUADOR GMS 429", "GMS429")]:
    normalized, is_valid = normalize_plate(raw)
    assert is_valid and normalized == expected, f"{raw!r} → {normalized} (se esperaba {expected})"
# Lecturas que deben quedar inválidas (→ verificación manual) en vez de convertirse en otra placa válida.
for raw in ["GTT-187G", "GPA-1G88", "OBX=660", "GBΦ6184", "115220", "ECUADOR", ""]:
    normalized, is_valid = normalize_plate(raw)
    assert not is_valid, f"{raw!r} no debería validar (salió {normalized})"
assert plate_confidence("ECUADOR\nGMS-429", [0.2] * 8 + [0.9, 0.8, 0.95, 0.3, 0.99, 0.7, 0.9]) == 0.7
for raw, expected in [("gba1234", "GBA-1234"), ("GBA-1234", "GBA-1234"), ("GMS429", "GMS-0429"), ("gms-429", "GMS-0429"),
                      ("PBA 123", "PBA-0123"), ("PBA-0123", "PBA-0123"), ("G8A1234", "G8A1234"), ("", "")]:
    assert format_plate(raw) == expected, f"{raw!r} → {format_plate(raw)!r} (se esperaba {expected!r})"
print("normalize_plate y format_plate: los casos de prueba pasan.")


### 2.2 Referencia corregida de SAVH (revisión de placas)

La hoja de Google **Revisión placas SAVH (142 legibles)**, en el repositorio de GitHub, tiene una fila por cada placa legible cuya lectura de la cámara no sigue el formato ecuatoriano. La columna `sugerencia` la propuso Claude el 16/10/2026 con reglas de formato y una revisión visual de cada imagen: 118 sugerencias coinciden con la imagen, 15 se corrigieron mirándola (la regla sola fallaba, por ejemplo `BX4948` → `PBX4948`) y 9 quedaron dudosas (filas con `n` = 7–9, 20–22 y 71–73 en la hoja). Son propuestas: la placa definitiva es la que confirme una persona mirando la imagen.

**Para confirmar**, en la última columna de la hoja se escribe una de estas respuestas. Basta con una fila por grupo de copias idénticas (columna `igual_a`): la respuesta se aplica a todo el grupo.

| Respuesta en la hoja | Placa que se usa | `estado_gt` |
|---|---|---|
| `ok` | la sugerencia | `confirmada` |
| una placa, p. ej. `GBU-1825` | esa placa | `corregida_por_revisor` |
| `ilegible` | ninguna: se excluye del EMR | `excluida_por_revisor` |
| vacía | la sugerencia, por ahora | `sugerida_sin_confirmar` |

Las filas que se agregaron a la hoja por desacuerdos entre la cámara y el OCR no traen sugerencia: ahí `ok` confirma la lectura de la cámara (`texto_actual`), y mientras no tengan respuesta la referencia sigue siendo esa lectura.

**Ojo con la O y la Q en la tercera letra.** La cámara escribe 0 para las dos y las sugerencias siempre proponen O, pero en la fila con `n` = 74 fast-plate-ocr leyó Q, y la misma placa de la fila 83 aparece en otras dos imágenes como `GSQ8533` según la cámara. Conviene mirar con cuidado las filas 74 y 83–85.

Las demás legibles quedan como `camara_formato_valido` (tienen formato válido y, en una muestra de 16, 15 coincidían con la imagen y 1 era dudosa) y las no legibles como `no_legible`. La ejecución desde GitHub usa la copia CSV versionada del repositorio; no lee Google Sheets ni requiere autenticar Google Drive. La referencia corregida se guarda en `WORK_DIR/ocr/savh_ground_truth_corregido.csv`, junto con una copia de la referencia usada, para que el informe cite datos fijos. Las copias idénticas se marcan para contar cada imagen una vez. Las dos celdas siguientes arman la referencia y muestran las imágenes que todavía no tienen respuesta.

Si la celda se corre sola en una sesión nueva, avisa qué celdas faltan: la sección 0, la extracción de SAVH (1.3) y la normalización (2.1).


In [ ]:
_missing = [name for name in ("OCR_RESULTS_DIR", "REVIEW_SHEET_ID", "REVIEW_CSV_FALLBACK", "PLATE_PATTERNS",
                              "clean_plate_text", "savh_gt", "SAVH_DIR") if name not in globals()]
if _missing:
    raise RuntimeError("Faltan variables de celdas anteriores: " + ", ".join(_missing) + ". En una sesión nueva, corre "
                       "primero la sección 0 completa, la extracción de SAVH (1.3) y la normalización (2.1).")

# 2.2 · Referencia corregida de SAVH
REVIEW_PLATE_COLUMN = "revision: ok o placa correcta"
REVIEW_OK = {"ok", "si", "sí"}
REVIEW_EXCLUDE = {"ilegible", "no legible", "excluir"}
CORRECTED_GT_PATH = OCR_RESULTS_DIR / "savh_ground_truth_corregido.csv"
REVIEW_SNAPSHOT_PATH = OCR_RESULTS_DIR / "revision_placas_142_leida.csv"  # la hoja tal como se leyó
STATE_PRIORITY = ["corregida_por_revisor", "excluida_por_revisor", "confirmada", "sugerida_sin_confirmar",
                  "camara_formato_valido", "no_legible"]  # dentro de un grupo de copias gana la respuesta más informativa


def has_plate_format(text: str) -> bool:
    """True si el texto (sin guion) es 3 letras + 3 o 4 dígitos (PLATE_PATTERNS)."""
    return any(len(text) == letters + digits and text[:letters].isalpha() and text[letters:].isdigit()
               for letters, digits in PLATE_PATTERNS)


def read_review_sheet() -> pd.DataFrame:
    """La hoja de revisión, leída con la cuenta de Google de Colab o, si no se puede, desde la copia en CSV."""
    if USE_GOOGLE_SHEET and REVIEW_SHEET_ID:
        try:
            from google.colab import auth
            import gspread
            from google.auth import default
            auth.authenticate_user()
            credentials, _ = default()
            values = gspread.authorize(credentials).open_by_key(REVIEW_SHEET_ID).sheet1.get_all_values()
            review = pd.DataFrame(values[1:], columns=values[0])
            print(f"Hoja de revisión leída desde Google Sheets: {len(review)} filas.")
        except Exception as error:
            if not REVIEW_CSV_FALLBACK.exists():
                raise FileNotFoundError(f"no se pudo leer la hoja ({type(error).__name__}) y no existe {REVIEW_CSV_FALLBACK}.") from error
            review = pd.read_csv(REVIEW_CSV_FALLBACK, dtype=str, keep_default_na=False)
            print(f"No se pudo leer Google Sheets ({type(error).__name__}); se usa el CSV versionado.")
    else:
        if not REVIEW_CSV_FALLBACK.exists():
            raise FileNotFoundError(f"no existe el CSV versionado de revisión: {REVIEW_CSV_FALLBACK}")
        review = pd.read_csv(REVIEW_CSV_FALLBACK, dtype=str, keep_default_na=False)
        print(f"Revisión leída desde GitHub: {len(review)} filas.")
    review = review.astype(str).apply(lambda column: column.str.strip())
    missing = {"n", "fila_gt", "imagen", "texto_actual", "sugerencia", "chequeo_visual", REVIEW_PLATE_COLUMN} - set(review.columns)
    if missing:
        raise ValueError(f"a la hoja de revisión le faltan columnas: {sorted(missing)}")
    return review


def build_corrected_reference(gt: pd.DataFrame, review: pd.DataFrame) -> tuple:
    """Aplica la revisión a ground_truth.csv (por fila_gt) y marca las copias idénticas de una misma imagen."""
    ref = gt.rename(columns={"plate_gt": "plate_camara"}).set_index("fila_gt")
    ref["plate_gt"] = ref["plate_camara"]
    ref["estado_gt"] = np.where(ref["folder"] == "legible", "camara_formato_valido", "no_legible")
    ref["n_hoja"], ref["chequeo_visual"], ref["respuesta"] = "", "", ""
    warnings = []
    for rec in review.to_dict("records"):
        row = int(rec["fila_gt"])
        if row not in ref.index or ref.at[row, "plate_camara"] != rec["texto_actual"]:
            warnings.append(f"fila_gt {row} ({rec['texto_actual']}) no coincide con ground_truth.csv; se ignora.")
            continue
        if Path(ref.at[row, "image_path"]).name != Path(rec["imagen"]).name:
            warnings.append(f"fila_gt {row}: el nombre de imagen de la hoja no coincide; se usa fila_gt.")
        answer = rec[REVIEW_PLATE_COLUMN].lower().strip(" .")
        suggestion = clean_plate_text(rec["sugerencia"]) or clean_plate_text(rec["texto_actual"])  # sin sugerencia: la cámara
        if answer in REVIEW_OK:
            plate, state = suggestion, "confirmada"
        elif answer in REVIEW_EXCLUDE:
            plate, state = "", "excluida_por_revisor"
        elif answer:
            plate, state = clean_plate_text(answer), "corregida_por_revisor"
        elif rec["sugerencia"]:
            plate, state = suggestion, "sugerida_sin_confirmar"
        else:  # agregada por desacuerdo con el OCR y todavía sin respuesta: sigue la lectura de la cámara
            plate, state = suggestion, "camara_formato_valido"
        if state != "excluida_por_revisor" and not has_plate_format(plate):
            warnings.append(f"fila_gt {row}: «{plate}» no tiene formato de placa ecuatoriana.")
        ref.loc[row, ["plate_gt", "estado_gt", "n_hoja", "chequeo_visual", "respuesta"]] = [
            plate, state, rec["n"], rec["chequeo_visual"], answer]
    ref = ref.reset_index()

    # Copias idénticas: cada grupo cuenta una vez y todo el grupo toma la respuesta más informativa.
    first_by_digest, group = {}, []
    for row, path in zip(ref["fila_gt"], ref["image_path"]):
        digest = hashlib.sha256((SAVH_DIR / path).read_bytes()).hexdigest()
        group.append(first_by_digest.setdefault(digest, row))
    ref["duplicado_de_fila"] = pd.array([g if g != r else pd.NA for g, r in zip(group, ref["fila_gt"])], dtype="Int64")
    priority = ref["estado_gt"].map({state: i for i, state in enumerate(STATE_PRIORITY)})
    best = (ref.assign(grupo=group, prioridad=priority).sort_values(["grupo", "prioridad"])
            .drop_duplicates("grupo").set_index("grupo"))
    ref["plate_gt"] = [best.at[g, "plate_gt"] for g in group]
    ref["estado_gt"] = [best.at[g, "estado_gt"] for g in group]
    return ref, warnings


savh_ref = None
if savh_gt is not None:
    try:
        review = read_review_sheet()
    except (FileNotFoundError, ValueError) as error:
        print("Aviso: sin revisión de placas,", error)
        print("Se salta la evaluación de SAVH: con la lectura de la cámara como referencia, el EMR no sería confiable.")
    else:
        review.to_csv(REVIEW_SNAPSHOT_PATH, index=False)
        savh_ref, warnings = build_corrected_reference(savh_gt, review)
        for warning in warnings:
            print("Aviso:", warning)
        savh_ref.to_csv(CORRECTED_GT_PATH, index=False)
        unique = savh_ref["duplicado_de_fila"].isna()
        display(pd.crosstab(savh_ref["estado_gt"], unique.map({True: "imágenes únicas", False: "copias"}),
                            margins=True, margins_name="total"))
        excluded = savh_ref["estado_gt"] == "excluida_por_revisor"
        changed = int(((savh_ref["plate_gt"] != savh_ref["plate_camara"]) & unique & ~excluded).sum())
        print(f"{changed} imágenes únicas cambian de placa respecto a la cámara y {int((unique & excluded).sum())} "
              f"quedan excluidas por el revisor. Referencia guardada en {CORRECTED_GT_PATH}")


In [ ]:
# 2.2 · Imágenes de la hoja de revisión que todavía no tienen respuesta
SHOW_PENDING_REVIEW = True  # imágenes de la hoja que todavía no tienen respuesta, una por grupo de copias

if SHOW_PENDING_REVIEW and savh_ref is not None:
    unique = savh_ref["duplicado_de_fila"].isna()
    added = (savh_ref["estado_gt"] == "camara_formato_valido") & (savh_ref["n_hoja"] != "")  # desacuerdos sin respuesta
    pending = savh_ref[unique & ((savh_ref["estado_gt"] == "sugerida_sin_confirmar") | added)]
    if pending.empty:
        print("Todas las placas de la hoja de revisión tienen respuesta.")
    else:
        pending = pending.assign(orden=pd.to_numeric(pending["n_hoja"], errors="coerce")).sort_values("orden")
        print(f"{len(pending)} imágenes por confirmar en la hoja de revisión (el número es la columna n de la hoja; en rojo, "
              "las dudosas; en azul, los desacuerdos con el OCR, con la lectura de la cámara).")
        pending = pending.head(min(8, int(SAVH_SAMPLE_SIZE or 8)))  # máximo ocho imágenes visibles por revisión
        columns = 4
        rows = -(-len(pending) // columns)
        fig, axes = plt.subplots(rows, columns, figsize=(2.7 * columns, 1.7 * rows), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, rec in zip(axes.flat, pending.itertuples(index=False)):
            ax.imshow(Image.open(SAVH_DIR / rec.image_path).convert("RGB"))
            doubtful = rec.chequeo_visual == "dudosa"
            disagreement = rec.estado_gt == "camara_formato_valido"
            tag = " · dudosa" if doubtful else (" · desacuerdo" if disagreement else "")
            ax.set_title(f"n {rec.n_hoja} · {rec.plate_gt}{tag}", fontsize=8.5,
                         color="#b91c1c" if doubtful else ("#1d4ed8" if disagreement else "#0b0b0b"))
        plt.tight_layout()
        plt.show()


In [ ]:
# Etiquetas revisadas para una futura partición OCR; no son escenas para el detector.
# No mezclar estos recortes con un test reservado: separar por vehículo/sesión y hash.
ocr_reviewed=pd.DataFrame()
if savh_ref is not None:
    accepted=savh_ref['estado_gt'].isin(['confirmada','corregida_por_revisor'])
    ocr_reviewed=savh_ref[accepted & savh_ref['duplicado_de_fila'].isna()].copy()
    ocr_reviewed=ocr_reviewed[ocr_reviewed['plate_gt'].map(has_plate_format)]
    ocr_reviewed.to_csv(OCR_RESULTS_DIR/'ecuador_ocr_etiquetas_revisadas.csv',index=False)
    print('Recortes únicos con transcripción confirmada:',len(ocr_reviewed))


## 3. Modelado y decisión del experimento
### 3.1 Entrenar RT-DETRv2-S con escenas ecuatorianas (opcional)

Con los datos COCO revisados, activa `RUN_TRAIN_ECUADOR=True` y vuelve a ejecutar esta
celda. Máximo 30 épocas, paciencia 8, semilla 2; selecciona el checkpoint con menor
pérdida en validación. El test no interviene. Guarda Transformers y OpenVINO en
`WORK_DIR/models/detector/rtdetrv2_s/seed_2/`, sin cambiar el checkpoint privado USA ni el de la aplicación.

El IR OpenVINO sirve para inferir; para continuar un entrenamiento USA se necesitan
sus pesos Transformers originales. Si `TRAINING_MODEL_SOURCE` sigue con su valor inicial,
el ajuste empieza en el checkpoint general de placas indicado, no en los pesos USA.
La celda adapta los argumentos de Transformers a la versión instalada y no depende
incondicionalmente de `warmup_ratio` o `save_safetensors`.

La selección usa pérdida de validación, no mAP. La prueba independiente y la calibración Ecuador quedan pendientes hasta ejecutarlas con sus referencias.


In [ ]:
def training_arguments_ecuador(TrainingArguments,total_steps):
    supported=inspect.signature(TrainingArguments).parameters
    values=dict(output_dir=str(TRAIN_RUN_DIR),num_train_epochs=EPOCHS,
        learning_rate=5e-5,weight_decay=1e-4,per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,save_strategy='epoch',save_total_limit=2,
        load_best_model_at_end=True,metric_for_best_model='eval_loss',greater_is_better=False,
        remove_unused_columns=False,dataloader_num_workers=2,seed=TRAIN_SEED,
        data_seed=TRAIN_SEED,report_to='none',logging_steps=25)
    values['eval_strategy' if 'eval_strategy' in supported else 'evaluation_strategy']='epoch'
    if 'warmup_ratio' in supported: values['warmup_ratio']=0.05
    elif 'warmup_steps' in supported: values['warmup_steps']=max(1,math.ceil(total_steps*0.05))
    if 'save_safetensors' in supported: values['save_safetensors']=True
    values['prediction_loss_only'] = True
    if 'eval_do_concat_batches' in supported: values['eval_do_concat_batches'] = False
    return values

if RUN_TRAIN_ECUADOR:
    import torch
    from transformers import AutoImageProcessor,RTDetrV2ForObjectDetection,TrainingArguments,Trainer,EarlyStoppingCallback
    from src.train import CocoPlateDataset,detection_collator
    from src.export_openvino import export_openvino_model
    display(validate_ecuador_coco(ECUADOR_COCO_DIR))
    if (MODEL_OUTPUT_DIR/'openvino/openvino_model.xml').exists():
        raise FileExistsError('Ya existe un candidato Ecuador. Usa otra MODEL_OUTPUT_DIR y TRAIN_RUN_DIR para una nueva corrida.')
    torch.manual_seed(TRAIN_SEED)
    processor=AutoImageProcessor.from_pretrained(str(TRAINING_MODEL_SOURCE),trust_remote_code=False,use_fast=False)
    model=RTDetrV2ForObjectDetection.from_pretrained(str(TRAINING_MODEL_SOURCE),trust_remote_code=False)
    if model.config.num_labels!=1: raise ValueError('El checkpoint inicial debe tener una sola clase de placa.')
    model.config.id2label={0:'license_plate'};model.config.label2id={'license_plate':0}
    train_data=CocoPlateDataset(ECUADOR_COCO_DIR,'train',processor)
    val_data=CocoPlateDataset(ECUADOR_COCO_DIR,'val',processor)
    total_steps=math.ceil(len(train_data)/BATCH_SIZE)*EPOCHS
    options=training_arguments_ecuador(TrainingArguments,total_steps)
    options['fp16']=torch.cuda.is_available()
    args=TrainingArguments(**options)
    trainer=Trainer(model=model,args=args,train_dataset=train_data,eval_dataset=val_data,
        data_collator=partial(detection_collator,image_processor=processor),processing_class=processor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    transformers_dir=MODEL_OUTPUT_DIR/'transformers'
    trainer.save_model(str(transformers_dir));processor.save_pretrained(str(transformers_dir))
    manifest={'country':'ecuador','architecture':'RT-DETRv2-S','seed':TRAIN_SEED,
        'training_source':str(TRAINING_MODEL_SOURCE),'dataset_dir':str(ECUADOR_COCO_DIR),
        'epochs_requested':EPOCHS,'early_stopping_patience':PATIENCE,
        'checkpoint_selection':'validation eval_loss','best_checkpoint':trainer.state.best_model_checkpoint,
        'detector_threshold':DETECTOR_THRESHOLD,'ocr_threshold':OCR_THRESHOLD,'independent_test_evaluated':False}
    openvino_dir=export_openvino_model(transformers_dir,MODEL_OUTPUT_DIR/'openvino',image_size=640)
    for directory in (transformers_dir,openvino_dir):
        (directory/'sirap_training.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Candidato Ecuador guardado en',MODEL_OUTPUT_DIR,'; no se publica ni activa automáticamente.')
else:
    print('Entrenamiento preparado. RUN_TRAIN_ECUADOR=False: no se entrenó ni se sustituyeron pesos.')


### 3.2 Selección del modelo y requisitos de cámara

Esta celda define la selección; **la carga se hace en 4**. Con `USE_ECUADOR_TRAINED_MODEL=True`
se busca primero el candidato de 3.1 en `WORK_DIR/models/detector/rtdetrv2_s/seed_2/openvino/`
y después `models/ecuador/detector/rtdetrv2_s/openvino/` del clon GitHub. Se comprueban
XML, BIN, manifiesto OpenVINO y manifiesto de entrenamiento con `country="ecuador"`.

Si se desactiva esa selección, la configuración Ecuador puede cargar el checkpoint
compartido verificado de la aplicación. Su secreto conserva el nombre existente
`SIRAP_USA_MODEL_TOKEN`; configúralo en Secrets de Colab. Alternativamente,
`USE_BASE_MODEL_FOR_SMOKE=True` exporta el checkpoint inicial a una carpeta técnica
del espacio de trabajo Ecuador. Ninguna de esas dos alternativas se etiqueta como
entrenamiento o validación Ecuador, ni permite publicar un candidato Ecuador desde 9.

El video usa detección 0,70 y OCR mínimo 0,94 para reproducir la regla del experimento.
Las sesiones de reconocimiento en cámara requieren, además, un informe **obtenido
calibrando en validación Ecuador**, guardado en `results/ecuador/metrics/rtdetr_ecuador_threshold_calibration.json`.
Debe tener esta estructura (los valores son ilustrativos, no resultados):

```json
{
  "country": "ecuador",
  "usable_for_inference": true,
  "threshold": 0.70,
  "checkpoint_sha256": {
    "openvino_model.xml": "SHA256 real del XML evaluado",
    "openvino_model.bin": "SHA256 real del BIN evaluado"
  }
}
```

Usa el umbral y los hashes producidos por esa validación; no copies literalmente el
ejemplo. Se valida el contenido exacto del IR para que moverlo del espacio de trabajo
a GitHub o al PC de la cámara no invalide su identidad. El notebook no crea ese informe
sin una calibración y no usa métricas de USA para completarlo.


In [ ]:
pipeline = None
detector = None
MODEL_IS_COUNTRY_FINETUNED = False
MODEL_ORIGIN = 'not_loaded'
COUNTRY_IR_DIR = MODEL_OUTPUT_DIR / 'openvino'
COUNTRY_IR_XML = COUNTRY_IR_DIR / 'openvino_model.xml'
SHARED_IR_DIR = WORK_DIR / 'models/detector/technical_fallback/openvino'
DETECTOR_DIR = None
LIVE_THRESHOLD_RECORD_PATH = METRICS_DIR / 'rtdetr_ecuador_threshold_calibration.json'


def require_ecuador_ir(directory):
    """Comprueba los cuatro archivos del bundle y su identificación Ecuador."""
    directory = Path(directory)
    required = ('openvino_model.xml', 'openvino_model.bin', 'sirap_openvino.json', 'sirap_training.json')
    missing = [name for name in required if not (directory / name).is_file()]
    if missing:
        raise FileNotFoundError(f'IR Ecuador incompleto en {directory}: {missing}')
    manifest = json.loads((directory / 'sirap_training.json').read_text(encoding='utf-8'))
    if manifest.get('country') != 'ecuador':
        raise ValueError('El manifiesto de entrenamiento no identifica un candidato Ecuador.')
    return directory


def validate_live_calibration(directory, record_path):
    """Vincula el umbral validado de Ecuador a los bytes exactos del XML y BIN."""
    directory = require_ecuador_ir(directory)
    record_path = Path(record_path)
    if not record_path.is_file():
        raise FileNotFoundError(f'Falta la calibración Ecuador sobre validación: {record_path}. Ver 3.2.')
    record = json.loads(record_path.read_text(encoding='utf-8'))
    threshold = record.get('threshold')
    if (record.get('country') != 'ecuador' or record.get('usable_for_inference') is not True
            or isinstance(threshold, bool) or not isinstance(threshold, (int, float))
            or not math.isfinite(threshold) or not 0 <= threshold <= 1):
        raise ValueError('La calibración Ecuador no habilita inferencia o no tiene un umbral válido.')
    expected = record.get('checkpoint_sha256', {})
    for name in ('openvino_model.xml', 'openvino_model.bin'):
        actual = hashlib.sha256((directory / name).read_bytes()).hexdigest()
        if expected.get(name) != actual:
            raise ValueError(f'El informe de calibración no corresponde al IR seleccionado: {name}.')
    return float(threshold), 'calibración Ecuador sobre validación'


print('La carga del detector y OCR se ejecuta en 4 cuando RUN_INFERENCE=True.')
print('Candidato local:', COUNTRY_IR_DIR)
print('Candidato publicado:', REPO_MODEL_OUTPUT / 'openvino')
print('Informe requerido para cámara:', LIVE_THRESHOLD_RECORD_PATH)


### 3.3 Lista de placas autorizadas

Un CSV con una placa por línea local en `WORK_DIR/authorized_plates.csv`. Si no existe se crea vacío para que el pipeline no falle. **La lista autorizada se mantiene en este CSV**, una placa por línea. Las placas cercanas a una autorización pasan a verificación manual. La lista simulada con la que se evalúa la regla en SAVH (experimento separado) se mantendría aparte, en `ocr/authorized_plates_simulada.csv`, y no reemplaza a esta. Las placas se pasan al formato XXX-0000 con `format_plate`, así que `GBA-1234`, `gba1234` y `GBA 1234` cuentan como la misma, igual que `GMS-429` y `GMS-0429`.


In [ ]:
# 3.3 · Lista de placas autorizadas. format_plate y clean_plate_text están en la normalización (2.1).


def load_authorized_plates(path: Path) -> set:
    """Placas del CSV en formato XXX-0000; si el archivo no existe, lo crea vacío."""
    path.parent.mkdir(parents=True,exist_ok=True)
    if not path.exists():
        path.write_text("")
        print(f"Aviso: {path} no existía; se creó vacío. Agrega placas autorizadas, una por línea.")
    with open(path, newline="") as f:
        plates = {format_plate(plate) for row in csv.reader(f) if row and row[0].strip() for plate,valid in [normalize_plate(row[0])] if valid}  # XXX-0000
    return plates - {"PLACA", "PLATE"}  # por si el CSV trae encabezado


def edit_distance_leq_1(a: str, b: str) -> bool:
    """True si las placas están a un carácter de distancia (Levenshtein ≤ 1): posible lectura de una autorizada."""
    if abs(len(a) - len(b)) > 1:
        return False
    return Levenshtein.distance(a, b) <= 1


AUTHORIZED_PLATES = load_authorized_plates(AUTHORIZED_LIST_PATH)
print(f"{len(AUTHORIZED_PLATES)} placas autorizadas en {AUTHORIZED_LIST_PATH}")


### 3.4 Regla de decisión del experimento

La lista CSV se usa únicamente en este notebook. Confianza baja, formato inválido o
una coincidencia aproximada pasan a revisión manual. Ni el SRI ni el nombre del
propietario autorizan un acceso. Este experimento **no acciona una barrera** ni escribe
operaciones en la aplicación.


In [ ]:
def decidir_ecuador(reading):
    plate=reading.plate
    if not plate or not math.isfinite(reading.det_confidence) or not math.isfinite(reading.ocr_confidence) or reading.det_confidence<DETECTOR_THRESHOLD or reading.ocr_confidence<OCR_THRESHOLD:
        return 'VERIFICACION_MANUAL'
    if plate in AUTHORIZED_PLATES:return 'AUTORIZADO'
    if any(edit_distance_leq_1(plate,p) for p in AUTHORIZED_PLATES):return 'VERIFICACION_MANUAL'
    return 'NO_REGISTRADO'


## 4. Cargar el IR OpenVINO de Ecuador

Integrada de la sección 13 USA. Con `RUN_INFERENCE=True`, carga en CPU el IR Ecuador
seleccionado en 3.2 y prepara el OCR con patrones ecuatorianos. Un bundle local entrenado
en 3.1 tiene prioridad sobre el publicado. Las alternativas técnicas se eligen expresamente
en 0.2; nunca se presentan como entrenamiento Ecuador. La carga no publica ni activa pesos
en la aplicación. Para inferencia no se usan directamente `.pt`, `.safetensors` u ONNX del detector.


In [ ]:
from src.export_openvino import export_openvino_model
from src.model import PlateDetector
from src.usa_pipeline import USAPipeline  # Implementación común; country y patrones son Ecuador.

pipeline = None
detector = None
MODEL_IS_COUNTRY_FINETUNED = False
MODEL_ORIGIN = 'not_loaded'
DETECTOR_DIR = None

if RUN_INFERENCE:
    if USE_ECUADOR_TRAINED_MODEL:
        candidates = (MODEL_OUTPUT_DIR / 'openvino', REPO_MODEL_OUTPUT / 'openvino')
        COUNTRY_IR_DIR = next((path for path in candidates if (path / 'openvino_model.xml').is_file()), candidates[0])
        COUNTRY_IR_XML = COUNTRY_IR_DIR / 'openvino_model.xml'
        require_ecuador_ir(COUNTRY_IR_DIR)
        DETECTOR_DIR = COUNTRY_IR_DIR
        MODEL_IS_COUNTRY_FINETUNED = True
        MODEL_ORIGIN = 'ecuador_finetuned'
    elif USE_BASE_MODEL_FOR_SMOKE:
        DETECTOR_DIR = SHARED_IR_DIR
        if not (DETECTOR_DIR / 'openvino_model.xml').is_file():
            export_openvino_model(MODEL_ID, DETECTOR_DIR, image_size=IMAGE_SIZE)
        MODEL_ORIGIN = 'initial_checkpoint_technical_smoke'
    else:
        # Opción explícita: usa el modelo compartido verificado configurado para Ecuador.
        # El nombre del secreto es el del repositorio privado existente, no el país de la lectura.
        from src.utils import get_secret
        secret = get_secret('SIRAP_USA_MODEL_TOKEN')
        if secret:
            os.environ['SIRAP_USA_MODEL_TOKEN'] = secret
        pipeline = USAPipeline.from_config(COUNTRY_CONFIG).for_country('ecuador', PLATE_PATTERNS)
        detector = pipeline.detector
        DETECTOR_DIR = detector.model_path.parent
        MODEL_ORIGIN = 'shared_verified_checkpoint_ecuador_rules'

    if detector is None:
        detector = PlateDetector(checkpoint=DETECTOR_DIR, model_id=MODEL_ID, imgsz=IMAGE_SIZE,
                                 conf=DETECTOR_THRESHOLD, class_id=0, device='CPU', fallback_checkpoint=None)
        reader = RuntimeSelectedSerialReader(model_name=OCR_MODEL_NAME)
        pipeline = USAPipeline(detector, reader, country='ecuador', patterns=PLATE_PATTERNS)
    print('Modelo cargado:', detector.name)
    print('Entrenado específicamente con Ecuador:', MODEL_IS_COUNTRY_FINETUNED)
    print('Origen:', MODEL_ORIGIN)
    print('Dispositivos OpenVINO:', detector.core.available_devices)
else:
    print('Carga desactivada. Activa RUN_INFERENCE y ejecuta 4 para cargar el IR.')


## 5. Prueba básica de OpenVINO

Integrada de la sección 14 USA. Envía una imagen negra al detector cargado en 4 para
comprobar que el IR funciona sin excepciones. El número de cajas de una imagen negra
no mide exactitud, precisión ni rendimiento del sistema completo.


In [ ]:
if detector is None:
    print("Prueba pendiente: activa RUN_INFERENCE y ejecuta 4.")
else:
    blank_image = np.zeros((480, 640, 3), dtype=np.uint8)
    smoke_detections = detector.detect(blank_image, max_detections=20)
    print("OpenVINO inference completed.")
    print("Boxes returned for the blank image:", len(smoke_detections))
    print("This check only confirms that the model loads and runs; it does not measure accuracy.")


## 6. Sistema completo: pipeline de demostración en video
### 6.1 Fuente de video

Elige una grabación propia de Ecuador mediante `SIRAP_ECUADOR_VIDEO_PATH` o `VIDEO_PATH`.
Por defecto se busca `datasets/ecuador/videos/demo_ecuador.mp4` en el clon GitHub.
Ese archivo no se inventa ni se descarga de otro país si falta. Para un video propio de
Drive puedes asignar su ID a `VIDEO_ID` y activar `DOWNLOAD_DEMO_VIDEO`; se descargará
a `WORK_DIR/videos/`. Un video de demostración no constituye un test independiente.


In [ ]:
VIDEO_ID = ''  # ID de un video propio de Ecuador; vacío por defecto.
VIDEO_PATH = Path(os.environ.get('SIRAP_ECUADOR_VIDEO_PATH', str(DATASET_DIR / 'videos/demo_ecuador.mp4')))
DOWNLOAD_DEMO_VIDEO = False
if RUN_VIDEO and DOWNLOAD_DEMO_VIDEO:
    if not VIDEO_ID:
        raise ValueError('Configura VIDEO_ID con tu grabación Ecuador antes de descargar.')
    VIDEO_PATH = WORK_DIR / 'videos/demo_ecuador.mp4'
    VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not VIDEO_PATH.is_file():
        import gdown
        downloaded = gdown.download(id=VIDEO_ID, output=str(VIDEO_PATH), quiet=False)
        if not downloaded or not VIDEO_PATH.is_file():
            raise FileNotFoundError('No se pudo descargar el video Ecuador indicado.')


### 6.2 Registro y muestreo

Se analiza un cuadro cada 2,5 segundos, **máximo ocho cuadros** por defecto. Todas las
placas que detecta el pipeline se guardan con país, modelo y confianza; no se fabrican
transcripciones de las placas ilegibles. Los CSV permanecen en `WORK_DIR/registros_csv/`.


In [ ]:
VIDEO_SAMPLE_SECONDS=2.5
MAX_VIDEO_FRAMES=8
VIDEO_RESULTS_PATH=WORK_DIR/'decisions_rtdetr_ecuador.csv'


### 6.3 Autorizaciones CSV y verificación en el SRI

El pipeline obtiene las autorizaciones exclusivamente de `authorized_plates.csv`. La lista de placas autorizadas se mantiene en `authorized_plates.csv`, en la carpeta de trabajo local, una placa por línea. Si el archivo no existe, se crea vacío. Las placas se normalizan al formato `XXX-0000`; una placa autorizada se marca `AUTORIZADO`, una coincidencia a un carácter pasa a `VERIFICACION_MANUAL` y el resto queda `NO_REGISTRADO`.

Las lecturas, consultas SRI y consultas del nombre público se guardan en archivos CSV bajo `registros_csv/` en el espacio temporal de la sesión. La sincronización externa permanece desactivada. Esta integración no escribe CSV operativos en GitHub ni cambia las autorizaciones de la aplicación.

**SRI opcional.** Con `SRI_VERIFICAR=True`, cada placa válida se consulta en segundo plano con el servicio público de la página «Valores a pagar por placa»: marca, modelo, año, país, último año pagado y valores pendientes. Si la placa antigua con cero inicial no aparece, reintenta con sus tres dígitos. Como máximo se envía una consulta cada 2 segundos y se reutiliza una consulta exitosa durante 24 horas. Los errores se guardan en `registros_csv/consultas_sri.csv` para poder reintentar en otra corrida. El SRI no devuelve el nombre del propietario y su respuesta no cambia la decisión de acceso.

Los CSV de trabajo quedan en `WORK_DIR/registros_csv/` durante la sesión y no se publican automáticamente. La sección 0 solo monta Google Drive si se activa PERSIST_OUTPUTS_TO_DRIVE; los CSV SRI/propietario siguen siendo locales.


In [ ]:
# ===================== 6.3 · Archivos CSV + verificación automática en el SRI =====================
import base64
import csv
import datetime as dt
import io
import json
import os
import re
import threading
import time
import unicodedata
import uuid
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import quote

import requests

# SIRAP guarda la lista autorizada y los resultados del pipeline en archivos CSV.
# Los CSV de trabajo quedan en el espacio temporal de la sesión. Hemos desactivado por completo la sincronización con GitHub.
PIPELINE_CSV_DIR = WORK_DIR / "registros_csv"
PIPELINE_CSV_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATHS = {
    "lecturas": PIPELINE_CSV_DIR / "lecturas.csv",
    "consultas_sri": PIPELINE_CSV_DIR / "consultas_sri.csv",
    "propietarios_vehiculo": PIPELINE_CSV_DIR / "propietarios.csv",
    "historial_propietarios": PIPELINE_CSV_DIR / "historial_propietarios.csv",
    "consultas_propietario": PIPELINE_CSV_DIR / "consultas_propietario.csv",
}

SRI_VERIFICAR = False  # activar solo cuando quieras consultar el servicio
SRI_CONSULTAR_RUBROS = True
SRI_CACHE_HOURS = 24
SRI_MIN_INTERVAL_S = 2.0
SRI_TIMEOUT_S = 15
SRI_REST = "https://srienlinea.sri.gob.ec/sri-matriculacion-vehicular-recaudacion-servicio-internet/rest"
SRI_HEADERS = {"Accept": "application/json", "User-Agent": "ProyectoIntegrador-ALPR/1.0 (uso academico)"}



def now_ec() -> dt.datetime:
    """Hora de Ecuador (UTC-5), sin zona horaria para mantener el formato de los registros."""
    return dt.datetime.now(dt.timezone(dt.timedelta(hours=-5))).replace(tzinfo=None, microsecond=0)


def ascii_text(text) -> str:
    """OpenCV no dibuja tildes: 'LOGÍSTICA ÑANDÚ' → 'LOGISTICA NANDU'."""
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()


class CsvPipelineStore:
    """Persistencia en archivos CSV del espacio temporal de Colab."""

    def __init__(self):
        self.lock = threading.RLock()
        for table, path in CSV_PATHS.items():
            if not path.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_text("", encoding="utf-8")

    def read(self, table: str) -> pd.DataFrame:
        if table not in CSV_PATHS:
            raise ValueError(f"CSV desconocido: {table}")
        path = CSV_PATHS[table]
        if not path.exists() or path.stat().st_size == 0:
            return pd.DataFrame()
        try:
            return pd.read_csv(path, keep_default_na=False)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()

    def _write_rows(self, table: str, rows: list[dict]) -> None:
        path = CSV_PATHS[table]
        columns = list(dict.fromkeys([key for row in rows for key in row]))
        temp = path.with_suffix(path.suffix + ".tmp")
        with temp.open("w", encoding="utf-8-sig", newline="") as stream:
            if columns:
                writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore", lineterminator="\n")
                writer.writeheader()
                writer.writerows(rows)
        os.replace(temp, path)

    def insert(self, table: str, row: dict) -> None:
        if table not in CSV_PATHS:
            raise ValueError(f"CSV desconocido: {table}")
        with self.lock:
            current = self.read(table).to_dict("records")
            current.append({key: "" if value is None else value for key, value in row.items()})
            self._write_rows(table, current)

    def replace(self, table: str, rows: list[dict]) -> None:
        if table not in CSV_PATHS:
            raise ValueError(f"CSV desconocido: {table}")
        with self.lock:
            self._write_rows(table, rows)

    def latest_consulta(self, placa: str, max_age_hours: float):
        rows = self.read("consultas_sri")
        if rows.empty or "placa" not in rows or "consultado_en" not in rows:
            return None
        rows = rows[rows["placa"].astype(str).eq(placa)].copy()
        rows["_fecha"] = pd.to_datetime(rows["consultado_en"], errors="coerce")
        cutoff = now_ec() - dt.timedelta(hours=max_age_hours)
        rows = rows[rows["_fecha"].notna() & (rows["_fecha"] >= cutoff)]
        if "error" in rows:
            rows = rows[rows["error"].astype(str).eq("")]
        if rows.empty:
            return None
        record = rows.sort_values("_fecha").iloc[-1].drop(labels=["_fecha"]).to_dict()
        for key in ("encontrado", "matricula_al_dia"):
            if key in record and isinstance(record[key], str):
                record[key] = record[key].strip().lower() in ("true", "1", "sí", "si")
        for key in ("codigo_vehiculo", "ultimo_anio_pagado", "rubros_pendientes"):
            if key in record:
                try:
                    record[key] = int(float(record[key]))
                except (TypeError, ValueError):
                    record[key] = None
        for key in ("total_pendiente",):
            if key in record:
                try:
                    record[key] = float(record[key])
                except (TypeError, ValueError):
                    record[key] = None
        return record

    def owner(self, placa: str):
        rows = self.read("propietarios_vehiculo")
        if rows.empty or "placa" not in rows:
            return None
        matches = rows[rows["placa"].astype(str).eq(placa)]
        return None if matches.empty else matches.iloc[-1].to_dict()

    def owner_rows(self) -> list[dict]:
        return self.read("propietarios_vehiculo").to_dict("records")

    def upsert_owner(self, placa: str, nombre: str, fuente: str, fecha) -> tuple[str, str | None]:
        with self.lock:
            rows = self.read("propietarios_vehiculo").to_dict("records")
            existing = next((row for row in rows if str(row.get("placa")) == placa), None)
            anterior = existing.get("nombre") if existing else None
            if existing is None:
                rows.append({"placa": placa, "nombre": nombre, "fuente": fuente,
                             "consultado_en": fecha, "actualizado_en": fecha})
                estado = "insertado"
            elif anterior != nombre:
                existing.update({"nombre": nombre, "fuente": fuente,
                                 "consultado_en": fecha, "actualizado_en": fecha})
                estado = "actualizado"
            else:
                existing.update({"fuente": fuente, "consultado_en": fecha})
                estado = "sin_cambios"
            self._write_rows("propietarios_vehiculo", rows)
            if estado != "sin_cambios":
                self.insert("historial_propietarios", {"placa": placa, "cambiado_en": fecha,
                             "anterior": anterior, "nuevo": nombre, "fuente": fuente})
            return estado, anterior

    def clear_readings(self, corrida: str, video: str) -> None:
        rows = self.read("lecturas")
        if rows.empty:
            return
        keep = rows[~((rows.get("corrida", "").astype(str) == corrida)
                      & (rows.get("video", "").astype(str) == video))]
        self.replace("lecturas", keep.to_dict("records"))

    def verified_readings(self) -> pd.DataFrame:
        reads = self.read("lecturas")
        if reads.empty:
            return reads
        sri = self.read("consultas_sri")
        owners = self.read("propietarios_vehiculo")
        if not sri.empty and "placa" in sri:
            sri = sri.sort_values("consultado_en").drop_duplicates("placa", keep="last")
            rename = {key: f"sri_{key}" for key in sri.columns if key != "placa"}
            reads = reads.merge(sri.rename(columns=rename), on="placa", how="left")
        if not owners.empty and "placa" in owners:
            owners = owners.drop_duplicates("placa", keep="last").rename(columns={
                "nombre": "propietario_nombre", "fuente": "propietario_fuente",
                "consultado_en": "propietario_consultado_en", "actualizado_en": "propietario_actualizado_en"})
            reads = reads.merge(owners, on="placa", how="left")
        reads["registrado_csv"] = reads.get("autorizado_por_csv", False)
        return reads

    def save(self) -> None:
        """Conserva los CSV en el espacio temporal de Colab (sin sincronización con GitHub)."""
        try:
            self._sync_github()
        except Exception as error:
            print(f"CSV guardados en la carpeta temporal de la sesión; falló la sincronización con GitHub: {type(error).__name__}: {error}")

    def _sync_github(self) -> None:
        # Sincronización desactivada explícitamente a petición del usuario
        return


# La única fuente de placas autorizadas es el archivo CSV de excepciones/lista blanca.
# Las coincidencias a un carácter las manda a VERIFICACION_MANUAL en decide_access.
_csv_plates = load_authorized_plates(AUTHORIZED_LIST_PATH)
AUTHORIZED_PLATES = set(_csv_plates)
CSV_AUTORIZADAS = set(AUTHORIZED_PLATES)


def panel_detail(placa: str):
    if not placa:
        return None
    planta = "autorizado por CSV" if placa in AUTHORIZED_PLATES else "sin autorización en CSV"
    sri_text = sri.short(placa)
    return f"{planta} | {sri_text}" if sri_text else planta

_sri_session = requests.Session()
_sri_session.headers.update(SRI_HEADERS)


def _sri_get(url: str, params: dict):
    """GET con 2 reintentos si no hay conexión o el servidor falla (5xx)."""
    for attempt in range(3):
        try:
            response = _sri_session.get(url, params=params, timeout=SRI_TIMEOUT_S)
            if response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}", response=response)
            if response.status_code in (204, 404):
                return None, response.text
            response.raise_for_status()
            text = response.text.strip()
            return (json.loads(text) if text else None), text
        except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as error:
            is_server_error = error.response is None or error.response.status_code >= 500
            if attempt == 2 or not is_server_error:
                raise
            time.sleep(2 * (attempt + 1))


def consultar_sri(placa: str) -> dict:
    """Consulta el servicio público del SRI; no devuelve datos del propietario."""
    url = f"{SRI_REST}/BaseVehiculo/obtenerPorNumeroPlacaOPorNumeroCampvOPorNumeroCpn"
    found = lambda value: isinstance(value, dict) and bool(value.get("numeroPlaca") or value.get("codigoVehiculo"))
    compacta = placa.replace("-", "")
    vehiculo, texto = _sri_get(url, {"numeroPlacaCampvCpn": compacta})
    if not found(vehiculo) and re.fullmatch(r"[A-Z]{3}0[0-9]{3}", compacta):
        time.sleep(1)
        vehiculo, texto = _sri_get(url, {"numeroPlacaCampvCpn": compacta[:3] + compacta[4:]})
    resultado = {"placa": placa, "encontrado": False, "respuesta_vehiculo": texto[:4000] or None}
    if not found(vehiculo):
        resultado["alerta"] = "placa no encontrada en el SRI (¿lectura errónea del OCR?)"
        return resultado
    try:
        ultimo = int(vehiculo.get("ultimoAnioPagado"))
    except (TypeError, ValueError):
        ultimo = None
    resultado.update(
        encontrado=True, codigo_vehiculo=vehiculo.get("codigoVehiculo"), camv_cpn=vehiculo.get("numeroCamvCpn"),
        marca=vehiculo.get("descripcionMarca"), modelo=vehiculo.get("descripcionModelo"),
        anio_modelo=vehiculo.get("anioAuto"), pais=vehiculo.get("descripcionPais"), clase=vehiculo.get("nombreClase"),
        servicio=vehiculo.get("descripcionServicio"), canton=vehiculo.get("descripcionCanton"),
        estado_exoneracion=vehiculo.get("estadoExoneracion"), ultimo_anio_pagado=ultimo,
        matricula_al_dia=None if ultimo is None else ultimo >= now_ec().year,
        prohibido_enajenar=None if vehiculo.get("prohibidoEnajenar") is None else str(vehiculo["prohibidoEnajenar"]))
    alertas = [] if resultado["matricula_al_dia"] is not False else [f"sin pago del año {now_ec().year} (último: {ultimo})"]
    if SRI_CONSULTAR_RUBROS and vehiculo.get("codigoVehiculo"):
        try:
            rubros, texto_rubros = _sri_get(f"{SRI_REST}/ConsultaRubros/obtenerPorCodigoVehiculo",
                                            {"codigoVehiculo": vehiculo["codigoVehiculo"]})
            resultado["respuesta_rubros"] = texto_rubros[:4000] or None
            if rubros is None or isinstance(rubros, list):
                items = rubros or []
                resultado["rubros_pendientes"] = len(items)
                valores = [item.get("valor") for item in items if isinstance(item, dict)]
                if items and len(valores) == len(items) and all(isinstance(value, (int, float)) for value in valores):
                    resultado["total_pendiente"] = float(sum(valores))
                if items:
                    alertas.append("1 valor pendiente de pago" if len(items) == 1 else f"{len(items)} valores pendientes de pago")
        except Exception as error:
            resultado["respuesta_rubros"] = f"error: {type(error).__name__}: {error}"[:500]
    resultado["alerta"] = "; ".join(alertas) or None
    return resultado


class SRIVerifier:
    """Consulta el SRI en segundo plano y guarda la respuesta en CSV."""

    def __init__(self, store_: CsvPipelineStore):
        self.store = store_
        self.pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix="sri")
        self.results, self.futures, self.lock, self.last_call = {}, {}, threading.Lock(), 0.0

    def submit(self, placa: str) -> None:
        if not SRI_VERIFICAR or not placa:
            return
        with self.lock:
            if placa in self.results or placa in self.futures:
                return
            cached = self.store.latest_consulta(placa, SRI_CACHE_HOURS)
            if cached is not None:
                self.results[placa] = cached
                return
            self.futures[placa] = self.pool.submit(self._consultar, placa)

    def _consultar(self, placa: str) -> dict:
        espera = SRI_MIN_INTERVAL_S - (time.monotonic() - self.last_call)
        if espera > 0:
            time.sleep(espera)
        self.last_call = time.monotonic()
        try:
            resultado = consultar_sri(placa)
        except Exception as error:
            resultado = {"placa": placa, "encontrado": None, "error": f"{type(error).__name__}: {error}"[:500]}
        resultado["consultado_en"] = now_ec()
        self.store.insert("consultas_sri", resultado)
        with self.lock:
            self.results[placa] = resultado
            self.futures.pop(placa, None)
        return resultado

    def wait(self) -> None:
        with self.lock:
            jobs = list(self.futures.values())
        for future in jobs:
            future.result()

    def summary(self, placa: str):
        if not SRI_VERIFICAR or not placa:
            return None
        result = self.results.get(placa)
        if result is None:
            return "SRI: consultando..." if placa in self.futures else None
        if result.get("error"):
            return "SRI: sin respuesta"
        if not result.get("encontrado"):
            return "SRI: placa no encontrada"
        vehicle = ascii_text(f"{result.get('marca') or ''} {result.get('modelo') or ''}").strip()[:26]
        estado = "al dia" if result.get("matricula_al_dia") else f"ultimo pago {result.get('ultimo_anio_pagado')}"
        return f"{vehicle} {result.get('anio_modelo') or ''} | {estado}"

    def short(self, placa: str):
        if not SRI_VERIFICAR or not placa:
            return None
        result = self.results.get(placa)
        if result is None:
            return "SRI consultando..." if placa in self.futures else None
        if result.get("error"):
            return "SRI sin respuesta"
        if not result.get("encontrado"):
            return "SRI: placa no encontrada"
        marca = ascii_text(result.get("marca") or "").strip()[:14]
        estado = "al dia" if result.get("matricula_al_dia") else f"ult. pago {result.get('ultimo_anio_pagado')}"
        return f"SRI {marca} {estado}".replace("  ", " ")


def guardar_lectura(reading,frame_index,seconds,corrida,video):
    """Registro local de RT-DETR Ecuador; no hay campos ni modelos YOLO."""
    store.insert('lecturas',dict(event_id=str(uuid.uuid4()),registrado_en=now_ec(),
        modelo=reading.detector,country='ecuador',corrida=corrida,video=Path(video).name,
        frame_index=frame_index,segundo_video=seconds,placa=reading.plate or '',
        texto_ocr=reading.plate_raw,formato_valido=bool(reading.plate),
        ocr_conf=reading.ocr_confidence,det_conf=reading.det_confidence,
        estado_acceso=decidir_ecuador(reading),autorizado_por_csv=reading.plate in AUTHORIZED_PLATES))


store = CsvPipelineStore()
sri = SRIVerifier(store)
print(f"Autorizaciones leídas de {AUTHORIZED_LIST_PATH.name}: {len(AUTHORIZED_PLATES)}")
print("Lecturas y consultas guardadas en CSV de la sesión:", PIPELINE_CSV_DIR)


### 6.4 Nombre del propietario (consulta pública opcional y registro local)

Esta parte conserva la integración del adjunto para consultar el formulario público de [EcuadorLegalOnline](https://www.ecuadorlegalonline.com/consultas/agencia-nacional-de-transito/consultar-a-quien-pertenece-un-vehiculo-por-placa-ant/) **solo si el proveedor permite esa consulta y el nombre está visible sin pagar**. No se comprobó aquí que el sitio siga ofreciendo ese dato; si está restringido, se mantiene el registro local. No abre «VER REPORTE COMPLETO» ni intenta obtener cédula o RUC, dirección o contacto. Si hay CAPTCHA, restricción, datos ocultos o errores, **no cambia** el dato que ya estaba guardado, y antes de guardar un nombre comprueba que la placa de la respuesta sea la que se consultó.

**Cómo se integra:** en el pipeline de video (6.6), cada placa leída con formato válido se consulta en segundo plano en el SRI y aquí, cada servicio en su propio hilo (los propietarios, de uno en uno). Se consulta también cuando la placa no aparece en la lista autorizada. La placa se normaliza a `XXX-0000`. Los resultados y cambios se guardan en CSV bajo `WORK_DIR/registros_csv/` durante la sesión:

- `propietarios.csv`: un registro vigente por placa. Si no existía, se inserta; si el nombre cambió, se actualiza en ese CSV cuando se consulta; si coincide, solo se actualiza `consultado_en`.
- `historial_propietarios.csv`: guarda el nombre anterior y el nuevo cuando hay un alta o un cambio real.
- `consultas_propietario.csv`: estado e incidencias de cada consulta.
- `lecturas.csv`, `consultas_sri.csv` y `propietarios.csv`: archivos que se combinan al generar el resumen de la corrida.

**Estado inicial:** `PROPIETARIO_WEB_ACTIVO = False`. Activarlo con `True` (celda siguiente) instala Playwright y Chromium, que necesitan acceso a Internet en Colab. Ponlo en `False` si el sitio no permite o restringe la automatización: el notebook no elude bloqueos ni pagos. `PROPIETARIO_CACHE_HORAS = 0` vuelve a consultar en cada corrida para detectar cambios. Cada resultado se escribe inmediatamente en su CSV temporal de la sesión; la sincronización con GitHub está desactivada y los CSV no se publican automáticamente.

**Consultas manuales:** `consultar_propietario_ahora('ABC0123')` consulta una placa y `consultar_propietarios_registrados(limite=10)` consulta los nombres de las placas que aparecen en `authorized_plates.csv` y todavía no tienen resultado guardado. Entre peticiones al sitio se esperan 12 s (`PROPIETARIO_INTERVALO_S`).


In [ ]:
# 6.4 · Navegador (Playwright + Chromium) para leer SOLO el nombre público del propietario.
# Ponlo en False si el sitio no permite consultas automáticas o deja de responder. Nunca se resuelven CAPTCHAs,
# se hacen pagos ni se saltan controles del sitio.
PROPIETARIO_WEB_ACTIVO = False

if PROPIETARIO_WEB_ACTIVO:
    import importlib.util
    import subprocess
    import sys
    if importlib.util.find_spec("playwright") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
    # «playwright install» no repite la descarga si Chromium ya está en la sesión.
    try:
        subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
        print("Navegador Playwright preparado para la consulta pública de nombre.")
    except subprocess.CalledProcessError as exc:
        PROPIETARIO_WEB_ACTIVO = False
        print("Consulta de propietarios desactivada (no fue posible preparar Chromium):", exc)


In [ ]:
# ===================== 6.4 · Nombre público del propietario y archivos CSV =====================
# Fuente: solo el campo «Nombre», si el proveedor lo permite y está visible sin pagar. No se abre el reporte completo ni se
# guardan cédulas, direcciones o teléfonos. PROPIETARIO_WEB_ACTIVO se fija en la celda anterior.
PROPIETARIO_CACHE_HORAS = 0       # 0 = volver a consultar en cada corrida (detecta cambios de dueño)
PROPIETARIO_INTERVALO_S = 12.0    # espera mínima entre consultas al sitio, de una en una
PROPIETARIO_TIMEOUT_MS = 25000    # tiempo máximo para cargar la página y recibir la respuesta

PROPIETARIO_URL = (
    "https://www.ecuadorlegalonline.com/consultas/agencia-nacional-de-transito/"
    "consultar-a-quien-pertenece-un-vehiculo-por-placa-ant/"
)

def placa_consulta_propietario(placa: str) -> str:
    """Placa en formato XXX-0000 para consultar y guardar; ValueError si no es una placa ecuatoriana válida."""
    compacta = re.sub(r"[^A-Z0-9]", "", str(placa).strip().upper())
    if not re.fullmatch(r"[A-Z]{3}[0-9]{3,4}", compacta):
        raise ValueError(f"Formato de placa inválido: {placa!r}")
    return compacta[:3] + "-" + compacta[3:].zfill(4)


def _nombre_visible(texto: str | None) -> str | None:
    """El nombre si es un texto visible y plausible; None si está vacío, oculto (xxxx), es otro rótulo o es demasiado largo."""
    if not texto:
        return None
    valor = re.sub(r"\s+", " ", str(texto)).strip()
    if (not valor or len(valor) > 250 or not re.search(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", valor)
            or re.fullmatch(r"[xX*\- .]+", valor)
            or re.match(r"^(?:c[eé]dula|ruc|direcci[oó]n|tel[eé]fono|due[nñ]o anterior|disponible|no disponible|oculto|ninguno|nombre$)", valor, re.I)):
        return None
    return valor


def extraer_nombre_publico(texto: str) -> str | None:
    """Lee solo la línea «Nombre» del bloque «Datos del propietario del vehículo» que muestra la página."""
    zona = re.search(r"DATOS DEL PROPIETARIO DEL VEH[IÍ]CULO(.*?)(?:VER REPORTE COMPLETO|¿No encuentra datos|$)",
                     texto, re.I | re.S)
    if not zona:
        return None
    lineas = [l.strip() for l in zona.group(1).splitlines() if l.strip()]
    for i, linea in enumerate(lineas):
        m = re.match(r"^Nombre\s*[:：]?\s*(.*)$", linea, re.I)
        if m:
            valor = m.group(1).strip() or (lineas[i + 1] if i + 1 < len(lineas) else "")
            return _nombre_visible(valor)
    return None


def _placa_de_resultado(page) -> str | None:
    """La placa que la página muestra como resultado («Placa consultada»), no la que se escribió en el buscador.

    Así no se guarda el nombre de un resultado de ejemplo, de una consulta anterior o de otra placa.
    """
    body = page.locator("body").inner_text(timeout=8000)
    m = re.search(r"Placa consultada\s*[:：]?\s*([A-Z]{3}[-\s]?[0-9]{3,4})\b", body, re.I)
    if m:
        return placa_consulta_propietario(m.group(1))
    # En algunas versiones de la página la placa va en un campo de solo lectura junto al rótulo «Placa consultada»,
    # que no aparece en el texto de la página.
    try:
        bloque = page.get_by_text("Placa consultada", exact=True).first.locator("xpath=..")
        for candidato in bloque.locator("input[readonly], input[disabled]").all():
            valor = candidato.input_value()
            try:
                return placa_consulta_propietario(valor)
            except ValueError:
                pass
    except Exception:
        pass
    return None


def consultar_nombre_publico_web(placa: str) -> dict:
    """Llena el formulario público con la placa y lee solo el nombre gratuito → {"estado", "nombre" o "detalle"}.

    No toca «VER REPORTE COMPLETO», CAPTCHAs, servicios privados ni pagos. Si no puede confirmar que la respuesta es de
    la placa consultada, no devuelve nombre.
    """
    from playwright.sync_api import sync_playwright
    placa = placa_consulta_propietario(placa)
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=True)
        try:
            page = browser.new_page(locale="es-EC", viewport={"width": 1360, "height": 900})
            page.goto(PROPIETARIO_URL, wait_until="domcontentloaded", timeout=PROPIETARIO_TIMEOUT_MS)
            entrada = page.locator('input[name*="placa" i], input[placeholder*="placa" i]')
            if entrada.count() == 0:
                entrada = page.get_by_label("Placa", exact=True)
            if entrada.count() == 0:
                return {"estado": "interfaz_no_reconocida", "detalle": "No se localizó el campo Placa"}
            entrada.first.fill(placa.replace("-", ""))
            consultar = page.get_by_role("button", name=re.compile(r"^Consultar$", re.I))
            if consultar.count() == 0:
                return {"estado": "interfaz_no_reconocida", "detalle": "No se localizó Consultar"}
            consultar.first.click(timeout=PROPIETARIO_TIMEOUT_MS)
            fin = time.monotonic() + PROPIETARIO_TIMEOUT_MS / 1000
            while time.monotonic() < fin:
                texto = page.locator("body").inner_text(timeout=8000)
                if re.search(r"captcha|acceso denegado|verifica que (eres|no eres)", texto, re.I):
                    return {"estado": "acceso_restringido", "detalle": "El sitio requiere verificación humana"}
                respondida = _placa_de_resultado(page)
                if respondida:
                    if respondida != placa:
                        return {"estado": "placa_no_coincide", "detalle": "La placa de la respuesta no coincide"}
                    nombre = extraer_nombre_publico(texto)
                    if nombre:
                        return {"estado": "encontrado", "nombre": nombre}
                    if re.search(r"no se encontr[oó]|sin resultados|veh[ií]culo no existe", texto, re.I):
                        return {"estado": "sin_datos", "detalle": "El sitio no devolvió propietario"}
                time.sleep(0.75)
            return {"estado": "sin_respuesta_verificable", "detalle": "No se confirmó placa y nombre visibles"}
        finally:
            browser.close()


def sincronizar_nombre_propietario(placa: str, nombre: str, fuente="ecuadorlegalonline_publico") -> str:
    """Guarda el nombre vigente y conserva cambios anteriores en archivos CSV."""
    placa = placa_consulta_propietario(placa)
    nombre = _nombre_visible(nombre)
    if not nombre:
        raise ValueError("Nombre vacío, oculto o inválido; no se modifica el CSV")
    estado, _ = store.upsert_owner(placa, nombre, fuente, now_ec())
    return estado


def registrar_consulta_propietario(placa, estado, detalle=None):
    """Una fila en consultas_propietario por cada consulta, con su estado (encontrado, sin_datos, error…)."""
    store.insert("consultas_propietario", {
        "placa": placa_consulta_propietario(placa), "consultado_en": now_ec(),
        "estado": estado, "detalle": str(detalle)[:300] if detalle else None,
        "fuente": "ecuadorlegalonline_publico"
    })


class PropietarioVerifier:
    """Consulta los propietarios en un hilo aparte, de uno en uno, para no frenar el video ni el SRI."""
    def __init__(self, store_: CsvPipelineStore):
        self.store = store_
        self.pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix="propietario")
        self.lock = threading.Lock()
        self.results, self.futures = {}, {}
        self.last_call = 0.0

    def nueva_corrida(self):
        """Al empezar una corrida olvida los resultados en memoria, para volver a consultar las placas."""
        self.wait()
        with self.lock:
            self.results.clear()

    def submit(self, placa: str):
        """Pone la placa en la cola si es válida y no se consultó ya en esta corrida; devuelve su tarea o None."""
        if not PROPIETARIO_WEB_ACTIVO:
            return None
        try:
            placa = placa_consulta_propietario(placa)
        except ValueError:
            return None
        with self.lock:
            if placa in self.futures or placa in self.results:
                return self.futures.get(placa)
            future = self.pool.submit(self._consultar, placa)
            self.futures[placa] = future
            return future

    def _consultar(self, placa: str):
        """Consulta una placa respetando el intervalo, guarda el nombre y registra la consulta."""
        resultado = {"estado": "error", "detalle": "No iniciada"}
        try:
            if PROPIETARIO_CACHE_HORAS > 0:
                reciente = self.store.owner(placa)
                if reciente:
                    consultado = pd.to_datetime(reciente.get("consultado_en"), errors="coerce")
                    if pd.notna(consultado) and consultado >= now_ec() - dt.timedelta(hours=PROPIETARIO_CACHE_HORAS):
                        resultado = {"estado": "en_cache"}
                        return resultado
            espera = PROPIETARIO_INTERVALO_S - (time.monotonic() - self.last_call)
            if espera > 0:
                time.sleep(espera)
            self.last_call = time.monotonic()
            resultado = consultar_nombre_publico_web(placa)
            if resultado["estado"] == "encontrado":
                resultado["accion_csv"] = sincronizar_nombre_propietario(placa, resultado["nombre"])
            registrar_consulta_propietario(placa, resultado["estado"], resultado.get("detalle"))
            # El CSV local se escribe al registrar el nombre; GitHub se sincroniza al cerrar el pipeline.
        except Exception as exc:
            resultado = {"estado": "error", "detalle": f"{type(exc).__name__}: {exc}"[:300]}
            try:
                registrar_consulta_propietario(placa, "error", resultado["detalle"])
            except Exception:
                pass
        finally:
            with self.lock:
                self.results[placa] = resultado
                self.futures.pop(placa, None)
        return resultado

    def wait(self):
        """Espera las consultas en curso. El diccionario cambia mientras terminan: se copia con el candado tomado."""
        with self.lock:
            trabajos = list(self.futures.values())
        for future in trabajos:
            future.result()


if "propietario" in globals() and getattr(propietario, "store", None) is not store:
    try:
        propietario.wait()
        propietario.pool.shutdown(wait=False)
    except Exception:
        pass
propietario = PropietarioVerifier(store)


def consultar_propietario_ahora(placa: str) -> dict:
    """Consulta una placa ahora (aunque ya se haya consultado en esta corrida) y devuelve el resultado ya guardado."""
    placa = placa_consulta_propietario(placa)
    if not PROPIETARIO_WEB_ACTIVO:
        local=store.owner(placa)
        return {"estado":"registro_local","nombre":local["nombre"]} if local else {"estado":"desactivado","detalle":"Sin propietario local; la consulta web es opcional."}
    # Para repetir la consulta se olvida el resultado en memoria.
    propietario.wait()
    with propietario.lock:
        propietario.results.pop(placa, None)
    tarea = propietario.submit(placa)
    return tarea.result() if tarea is not None else {"estado": "no_iniciado"}


def consultar_propietarios_registrados(limite: int | None = None):
    """Consulta el nombre de las placas de la planta que todavía no lo tienen (como máximo `limite`).

    Opcional: respeta el intervalo entre consultas al sitio, así que 10 placas tardan unos 2 minutos.
    """
    if not PROPIETARIO_WEB_ACTIVO:
        print("La consulta web está desactivada.")
        return pd.DataFrame()
    ya_consultadas = {str(row.get("placa", "")) for row in store.owner_rows()}
    placas = sorted(AUTHORIZED_PLATES - ya_consultadas)
    if limite is not None:
        placas = placas[:limite]
    trabajos = {p: propietario.submit(p) for p in placas}
    resultados = [{"placa": p, **(t.result() if t else {"estado": "no_iniciado"})} for p, t in trabajos.items()]
    return pd.DataFrame(resultados)


print("Propietarios: consulta del nombre público activa:", PROPIETARIO_WEB_ACTIVO)
print("Propietarios guardados en CSV:", len(store.owner_rows()))
print("Consultar placa puntual: consultar_propietario_ahora('ABC0123')")
print("Completar propietarios faltantes (opcional): consultar_propietarios_registrados(limite=10)")


### 6.5 Comprobación de la extracción del nombre (sin usar el sitio web)

Estas pruebas verifican, con textos de ejemplo, que solo se extrae el nombre de la línea visible «Nombre» y que los marcadores ocultos (`xxxxxx`) no se toman como propietario. **No** consultan el sitio ni modifican la base. Al procesar el video, las consultas externas solo se hacen si se activaron expresamente en 6.3/6.4. El SRI no aporta etiquetas ni propietarios para entrenar.


In [ ]:
# 6.5 · Comprobaciones locales sin abrir navegador, sin llamadas al proveedor y sin escribir datos externos.
_ejemplo_publico = """Placa consultada ABC0123
DATOS DEL PROPIETARIO DEL VEHÍCULO
Nombre
EMPRESA DE EJEMPLO
Cédula / RUC
🔒 Disponible
VER REPORTE COMPLETO"""
_ejemplo_oculto = """DATOS DEL PROPIETARIO DEL VEHÍCULO
Nombre xxxxxx xxxxxx xxxxxx
Cédula / RUC
VER REPORTE COMPLETO"""
assert extraer_nombre_publico(_ejemplo_publico) == "EMPRESA DE EJEMPLO"
assert extraer_nombre_publico(_ejemplo_oculto) is None
assert placa_consulta_propietario("ABC123") == "ABC-0123"
print("Correcto: la extracción local del texto de ejemplo devuelve el nombre y rechaza marcadores ocultos; no se consultó ningún sitio.")


In [ ]:
# 6.5 · Revisión de los archivos CSV de propietarios
for nombre, tabla in (("propietarios.csv", "propietarios_vehiculo"),
                      ("historial_propietarios.csv", "historial_propietarios"),
                      ("consultas_propietario.csv", "consultas_propietario")):
    datos = store.read(tabla)
    print(f"{nombre}: {len(datos)} registros")
    if not datos.empty:
        display(datos.head(10))


### 6.6 Ejecutar y guardar el flujo integrado

Activa `RUN_INFERENCE` y selecciona el modelo en 3.2 y cárgalo en 4; después activa `RUN_VIDEO`, elige la
ruta de un video propio y ejecuta esta celda. Mantiene un máximo de ocho cuadros.
Las consultas SRI/propietario son opcionales, están separadas del entrenamiento y no
alteran la decisión CSV. No se abre una barrera ni se publican los CSV.


In [ ]:
if RUN_VIDEO:
    import cv2
    if pipeline is None:raise RuntimeError('Activa RUN_INFERENCE y ejecuta 4 antes del video.')
    if not VIDEO_PATH.is_file():raise FileNotFoundError(f'Falta el video: {VIDEO_PATH}')
    cap=cv2.VideoCapture(str(VIDEO_PATH))
    if not cap.isOpened():raise ValueError('No se pudo abrir el video.')
    fps=cap.get(cv2.CAP_PROP_FPS) or 25
    step=max(1,round(fps*VIDEO_SAMPLE_SECONDS))
    corrida=str(uuid.uuid4());index=0;sampled=0;rows=[]
    try:
        while sampled<MAX_VIDEO_FRAMES:
            ok,frame=cap.read()
            if not ok:break
            if index%step==0:
                readings=pipeline.recognize_many(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
                if not readings:
                    rows.append(dict(frame_index=index,second=index/fps,country='ecuador',plate_pred='',status='SIN_DETECCION'))
                for reading in readings:
                    guardar_lectura(reading,index,index/fps,corrida,VIDEO_PATH)
                    rows.append(dict(frame_index=index,second=index/fps,country='ecuador',plate_pred=reading.plate,
                        det_confidence=reading.det_confidence,ocr_confidence=reading.ocr_confidence,
                        status=decidir_ecuador(reading)))
                    if reading.plate:sri.submit(reading.plate);propietario.submit(reading.plate)
                sampled+=1
            index+=1
    finally:cap.release()
    sri.wait();propietario.wait()
    pd.DataFrame(rows).to_csv(VIDEO_RESULTS_PATH,index=False)
    store.verified_readings().to_csv(PIPELINE_CSV_DIR/'lecturas_enriquecidas.csv',index=False)
    store.save()  # solo archivos temporales; sincronización externa desactivada
    print('Cuadros analizados:',sampled,'| predicciones:',VIDEO_RESULTS_PATH)
else:
    print('Video preparado; RUN_VIDEO=False. No hubo inferencia ni consultas externas.')


## 7. Cámara en vivo: Cloudflare o PC junto a la cámara

Integrada de la sección 16 USA, con las cinco subsecciones originales. RT-DETRv2-S
detecta con OpenVINO nativo y el OCR global se normaliza a placas de Ecuador. El puente
se ejecuta en un PC con acceso a la cámara; Colab puede leerlo mediante HTTPS/Cloudflare.
También se genera un script para hacer toda la inferencia en el PC de la cámara.

Las conexiones y el reconocimiento empiezan desactivados. Definir o escribir los scripts
no inicia procesos ni túneles. Las lecturas se confirman por repetición, formato y umbral OCR;
no se conecta una barrera ni se importa la lista operativa de autorizaciones. Se conserva
la comprobación de calibración y checkpoint para evitar usar un umbral de otro modelo.


### 7.1 Generar el puente para el PC de la cámara

Esta celda escribe `WORK_DIR/live_camera/bridge_camera.py`; si está activada la copia a
Drive, conserva otra en `generated_scripts/ecuador/live_camera/`. Descárgalo al PC que
alcanza la cámara. Allí instala `opencv-python` y ejecútalo con `python bridge_camera.py --source 0`
para una webcam, o configura `SIRAP_CAMERA_SOURCE` para una cámara IP o archivo.

El puente escucha en `127.0.0.1:8080` y expone `/snapshot.jpg`, `/video` (MJPEG) y `/status`.
Protege las tres rutas con `X-Camera-Token`; crea un token aleatorio en el archivo local
`~/.sirap_camera_token`. Copia ese token al Secret de Colab **CAMARA_TOKEN**.

Instala Cloudflare Tunnel siguiendo su [guía oficial](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/downloads/).
En el PC ejecuta `cloudflared tunnel --url http://127.0.0.1:8080` y guarda la URL HTTPS
base en el Secret **CAMARA_URL**. La URL de un Quick Tunnel cambia al reiniciarlo.
Para un túnel estable protegido por Access configura también **CF_ACCESS_CLIENT_ID** y
**CF_ACCESS_CLIENT_SECRET**; el lector envía esas cabeceras además del token del puente.
Consulta la [guía de túneles](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)
y las [reglas de conectividad](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/configure-tunnels/tunnel-with-firewall/)
si el túnel no alcanza el proceso local. No se escriben credenciales en el script ni en sus salidas.


In [ ]:
from pathlib import Path

BRIDGE_SCRIPT_PATH = WORK_DIR / "live_camera/bridge_camera.py"
BRIDGE_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)

BRIDGE_SCRIPT = r'''
import argparse
import cv2
import hmac
import json
import os
import secrets
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

class FrameHub:
    def __init__(self, source, max_side, quality, fps, loop):
        self.source = source
        self.max_side = max_side
        self.quality = quality
        self.delay = 1.0 / max(fps, 1)
        self.loop = loop
        self.lock = threading.Lock()
        self.jpeg = None
        self.frame_id = 0
        self.captured_at = None
        self.width = 0
        self.height = 0
        self.last_error = ""
        self.stop_event = threading.Event()

    def start(self):
        threading.Thread(target=self._capture_loop, daemon=True).start()

    def snapshot(self):
        with self.lock:
            return self.jpeg, self.captured_at, self.frame_id

    def status(self):
        with self.lock:
            return {
                "ready": self.jpeg is not None,
                "width": self.width,
                "height": self.height,
                "frame_id": self.frame_id,
                "captured_at": self.captured_at,
                "last_error": self.last_error,
            }

    def _capture_loop(self):
        while not self.stop_event.is_set():
            capture = cv2.VideoCapture(self.source)
            capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            if not capture.isOpened():
                with self.lock:
                    self.last_error = "OpenCV could not open the configured camera source."
                capture.release()
                self.stop_event.wait(2)
                continue

            source_is_file = isinstance(self.source, str) and Path(self.source).is_file()
            while not self.stop_event.is_set():
                started = time.monotonic()
                ok, frame = capture.read()
                if not ok:
                    if source_is_file and self.loop:
                        capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
                        continue
                    if not source_is_file:
                        with self.lock:
                            self.last_error = "Camera frame read failed; reconnecting."
                        break
                    with self.lock:
                        self.last_error = "The video file reached its end."
                    self.stop_event.set()
                    break

                height, width = frame.shape[:2]
                scale = min(1.0, self.max_side / max(height, width))
                if scale < 1.0:
                    frame = cv2.resize(
                        frame,
                        (max(1, int(width * scale)), max(1, int(height * scale))),
                        interpolation=cv2.INTER_AREA,
                    )
                ok, encoded = cv2.imencode(
                    ".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, self.quality]
                )
                if ok:
                    with self.lock:
                        self.jpeg = encoded.tobytes()
                        self.frame_id += 1
                        self.captured_at = time.time()
                        self.height, self.width = frame.shape[:2]
                        self.last_error = ""
                remaining = self.delay - (time.monotonic() - started)
                if remaining > 0:
                    self.stop_event.wait(remaining)
            capture.release()
            if not source_is_file and not self.stop_event.is_set():
                self.stop_event.wait(1)

class Handler(BaseHTTPRequestHandler):
    hub = None
    token = ""

    def log_message(self, format, *args):
        return

    def _authorized(self):
        supplied = self.headers.get("X-Camera-Token", "")
        return hmac.compare_digest(supplied, self.token)

    def _send_json(self, value, status=200):
        body = json.dumps(value).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if not self._authorized():
            self._send_json({"error": "unauthorized"}, 401)
            return

        if self.path == "/status":
            self._send_json(self.hub.status())
            return

        if self.path == "/snapshot.jpg":
            jpeg, captured_at, frame_id = self.hub.snapshot()
            if jpeg is None:
                self._send_json({"error": "camera frame is not ready"}, 503)
                return
            self.send_response(200)
            self.send_header("Content-Type", "image/jpeg")
            self.send_header("Content-Length", str(len(jpeg)))
            self.send_header("X-Frame-Id", str(frame_id))
            self.send_header("X-Captured-At", str(captured_at))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(jpeg)
            return

        if self.path == "/video":
            self.send_response(200)
            self.send_header("Content-Type", "multipart/x-mixed-replace; boundary=frame")
            self.send_header("Cache-Control", "no-store")
            self.send_header("Connection", "close")
            self.end_headers()
            try:
                last_id = -1
                while True:
                    jpeg, captured_at, frame_id = self.hub.snapshot()
                    if jpeg is None or frame_id == last_id:
                        time.sleep(0.02)
                        continue
                    last_id = frame_id
                    part = (
                        b"--frame\r\nContent-Type: image/jpeg\r\n"
                        + b"Content-Length: " + str(len(jpeg)).encode("ascii") + b"\r\n"
                        + b"X-Captured-At: " + str(captured_at).encode("ascii") + b"\r\n\r\n"
                        + jpeg + b"\r\n"
                    )
                    self.wfile.write(part)
                    self.wfile.flush()
            except (BrokenPipeError, ConnectionResetError, OSError):
                return
            return

        self._send_json({"error": "not found"}, 404)

def load_or_create_token(path):
    path = Path(path).expanduser()
    if path.exists():
        token = path.read_text(encoding="utf-8").strip()
        if token:
            return token, path
    path.parent.mkdir(parents=True, exist_ok=True)
    token = secrets.token_urlsafe(32)
    path.write_text(token + "\n", encoding="utf-8")
    try:
        path.chmod(0o600)
    except OSError:
        pass
    return token, path

def parse_source(value):
    return int(value) if value.isdigit() else value

def main():
    parser = argparse.ArgumentParser(description="Token-protected SIRAP camera bridge")
    parser.add_argument("--source", default=os.environ.get("SIRAP_CAMERA_SOURCE", "0"))
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=8080)
    parser.add_argument("--max-side", type=int, default=1920)
    parser.add_argument("--quality", type=int, default=85)
    parser.add_argument("--fps", type=float, default=15)
    parser.add_argument("--loop", action="store_true", help="Loop a local video file.")
    args = parser.parse_args()

    token, token_path = load_or_create_token(Path.home() / ".sirap_camera_token")
    hub = FrameHub(
        parse_source(args.source), max(320, args.max_side),
        min(100, max(30, args.quality)), max(1, args.fps), args.loop,
    )
    hub.start()
    Handler.hub = hub
    Handler.token = token
    server = ThreadingHTTPServer((args.host, args.port), Handler)
    server.daemon_threads = True
    print("Camera bridge listening on http://127.0.0.1:" + str(args.port))
    print("Token file:", token_path)
    print("Routes: /snapshot.jpg, /video, /status")
    print("Stop with Ctrl+C. Keep this terminal open while the tunnel is running.")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        pass
    finally:
        hub.stop_event.set()
        server.server_close()

if __name__ == "__main__":
    main()
'''

BRIDGE_SCRIPT_PATH.write_text(BRIDGE_SCRIPT, encoding="utf-8")
bridge_persistent_path = persist_file_to_drive(
    BRIDGE_SCRIPT_PATH,
    DRIVE_SCRIPTS_DIR / "live_camera" / "bridge_camera.py",
)
print("Camera bridge script created:", BRIDGE_SCRIPT_PATH)
print("Download it from the Colab Files panel and copy it to the camera PC.")
print("On that PC, install OpenCV with: python -m pip install opencv-python")


### 7.2 Configurar el lector y el reconocimiento Ecuador

Se definen el lector de último cuadro, los intervalos y la carpeta local de salida;
todavía no se conecta la cámara ni se cargan modelos adicionales. En modo `auto` se
intenta MJPEG y, si no entrega cuadros, se usan solicitudes de instantáneas. El modo
`opencv` permite un video o fuente RTSP/HLS que el equipo pueda alcanzar directamente.

`build_live_pipeline()` exige el IR Ecuador cargado en 4 y el informe descrito en 3.2.
Comprueba país, habilitación, umbral finito y hashes XML/BIN antes de abrir la cámara.
Sin informe válido no se usa un umbral provisional. Las placas se normalizan con las
funciones de 2.1; solo se confirman las válidas con confianza OCR mínima de 0,94.
Se conserva el texto OCR original y la placa normalizada por separado.

Las sesiones usan `WORK_DIR/live_camera/`; con Drive activado se copian a
`private/ecuador/camera_sessions/`. El módulo compartido `src.usa_pipeline` conserva
su nombre técnico en el repositorio; este notebook siempre usa país y reglas Ecuador.


In [ ]:
from pathlib import Path
import os
import json
import threading
import time
from urllib.parse import urlsplit, urlunsplit

import cv2
import numpy as np
import requests
import yaml
from IPython.display import display
from PIL import Image
from src.data_processing import clean_text, crop_with_margin
from src.model import PlateDetector

def read_colab_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return str(value)
    except Exception:
        pass
    return os.environ.get(name, "")

LIVE_SOURCE = read_colab_secret("CAMARA_URL").strip()
LIVE_TOKEN = read_colab_secret("CAMARA_TOKEN").strip()
CF_ACCESS_CLIENT_ID = read_colab_secret("CF_ACCESS_CLIENT_ID").strip()
CF_ACCESS_CLIENT_SECRET = read_colab_secret("CF_ACCESS_CLIENT_SECRET").strip()

LIVE_READ_MODE = "auto"      # auto, mjpeg, snapshot, or opencv
LIVE_MJPEG_WAIT_S = 6
LIVE_SNAPSHOT_FPS = 4
LIVE_DURATION_S = 120        # None means run until the cell is interrupted.
LIVE_START_TIMEOUT_S = 30
LIVE_STALE_S = 20
LIVE_MAX_PROCESS_FPS = 5
LIVE_MAX_CANDIDATES = 5
LIVE_DISPLAY_FPS = 2
LIVE_CONFIRMATIONS = 2
LIVE_CONFIRM_WINDOW_S = 3
LIVE_DUPLICATE_COOLDOWN_S = 15
LIVE_SAVE_VIDEO = False
LIVE_OUTPUT_DIR = WORK_DIR / "live_camera"
LIVE_DRIVE_OUTPUT_DIR = DRIVE_PRIVATE_CAMERA_DIR if PERSIST_OUTPUTS_TO_DRIVE else LIVE_OUTPUT_DIR
LIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class CameraUnavailable(RuntimeError):
    pass

def _camera_error(response):
    messages = {
        401: "The bridge rejected the camera token. Check the Colab Secret CAMARA_TOKEN.",
        403: "The request was denied by the bridge or Cloudflare Access. Check the configured Secrets.",
        404: "The bridge route was not found. Use the base tunnel URL in CAMARA_URL.",
        502: "Cloudflare could not reach the camera bridge. Confirm both processes are running.",
        530: "Cloudflare could not establish the requested connection. Check the tunnel and Access settings.",
    }
    return messages.get(
        response.status_code,
        "Camera request failed with HTTP " + str(response.status_code),
    )

class CameraReader:
    def __init__(self, source, token="", mode="auto", snapshot_fps=4,
                 access_client_id="", access_client_secret="", mjpeg_wait_s=6):
        self.source = source.strip()
        self.token = token
        self.mode = mode.lower()
        self.snapshot_fps = max(1.0, float(snapshot_fps))
        self.mjpeg_wait_s = max(1.0, float(mjpeg_wait_s))
        self.access_client_id = access_client_id
        self.access_client_secret = access_client_secret
        self.stop_event = threading.Event()
        self.lock = threading.Lock()
        self.frame = None
        self.received = 0
        self.last_frame_monotonic = None
        self.last_frame_epoch = None
        self.error = ""
        self.thread = None
        self.base_url = self._base_url(self.source)

    @staticmethod
    def _base_url(source):
        parts = urlsplit(source)
        if parts.scheme not in ("http", "https"):
            return source
        path = parts.path.rstrip("/")
        for suffix in ("/video", "/snapshot.jpg", "/status"):
            if path.endswith(suffix):
                path = path[:-len(suffix)]
                break
        return urlunsplit((parts.scheme, parts.netloc, path, "", "")).rstrip("/")

    def _headers(self):
        headers = {}
        if self.token:
            headers["X-Camera-Token"] = self.token
        if self.access_client_id:
            headers["CF-Access-Client-Id"] = self.access_client_id
        if self.access_client_secret:
            headers["CF-Access-Client-Secret"] = self.access_client_secret
        return headers

    def start(self):
        if not self.source:
            raise CameraUnavailable("No camera URL is configured. Add CAMARA_URL to Colab Secrets.")
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=3)

    def snapshot(self):
        with self.lock:
            if self.frame is None:
                return None, self.received, self.last_frame_monotonic, self.last_frame_epoch
            return self.frame.copy(), self.received, self.last_frame_monotonic, self.last_frame_epoch

    def _offer(self, frame):
        if frame is None or frame.size == 0:
            return
        with self.lock:
            self.frame = frame
            self.received += 1
            self.last_frame_monotonic = time.monotonic()
            self.last_frame_epoch = time.time()
            self.error = ""

    def _read_opencv(self):
        source = int(self.source) if self.source.isdigit() else self.source
        capture = cv2.VideoCapture(source)
        capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        if not capture.isOpened():
            capture.release()
            raise CameraUnavailable("OpenCV could not open the configured source.")
        try:
            while not self.stop_event.is_set():
                ok, frame = capture.read()
                if not ok:
                    raise CameraUnavailable("OpenCV stopped receiving frames from the source.")
                self._offer(frame)
        finally:
            capture.release()

    def _read_mjpeg(self):
        response = requests.get(
            self.base_url + "/video", headers=self._headers(),
            stream=True, timeout=(8, 4),
        )
        if response.status_code >= 400:
            message = _camera_error(response)
            response.close()
            raise CameraUnavailable(message)
        before = self.received
        started = time.monotonic()
        buffer = bytearray()
        jpeg_start = bytes([255, 216])
        jpeg_end = bytes([255, 217])
        try:
            for chunk in response.iter_content(chunk_size=2048):
                if self.stop_event.is_set():
                    break
                if not chunk:
                    continue
                buffer.extend(chunk)
                while True:
                    start = buffer.find(jpeg_start)
                    end = buffer.find(jpeg_end, start + 2) if start >= 0 else -1
                    if start < 0 or end < 0:
                        if len(buffer) > 5_000_000:
                            del buffer[:-2]
                        break
                    jpeg = bytes(buffer[start:end + 2])
                    del buffer[:end + 2]
                    frame = cv2.imdecode(np.frombuffer(jpeg, dtype=np.uint8), cv2.IMREAD_COLOR)
                    self._offer(frame)
                if self.received == before and time.monotonic() - started >= self.mjpeg_wait_s:
                    return False
        finally:
            response.close()
        return self.received > before

    def _read_snapshots(self):
        delay = 1.0 / self.snapshot_fps
        while not self.stop_event.is_set():
            started = time.monotonic()
            response = requests.get(
                self.base_url + "/snapshot.jpg",
                headers=self._headers(), timeout=(8, 10),
            )
            if response.status_code >= 400:
                message = _camera_error(response)
                response.close()
                raise CameraUnavailable(message)
            frame = cv2.imdecode(
                np.frombuffer(response.content, dtype=np.uint8), cv2.IMREAD_COLOR
            )
            response.close()
            if frame is None:
                raise CameraUnavailable("The snapshot endpoint did not return a valid JPEG.")
            self._offer(frame)
            self.stop_event.wait(max(0.0, delay - (time.monotonic() - started)))

    def _run(self):
        if self.mode == "opencv" or urlsplit(self.source).scheme not in ("http", "https"):
            try:
                self._read_opencv()
            except Exception as exc:
                self.error = str(exc)
            return

        while not self.stop_event.is_set():
            try:
                if self.mode == "snapshot":
                    self._read_snapshots()
                    return
                try:
                    received_mjpeg = self._read_mjpeg()
                except Exception as exc:
                    self.error = str(exc)
                    if self.mode != "auto":
                        return
                    received_mjpeg = False
                if self.mode == "mjpeg":
                    if not received_mjpeg:
                        self.error = "No MJPEG frames arrived before the timeout."
                        return
                    continue
                if not received_mjpeg:
                    self._read_snapshots()
                    return
            except Exception as exc:
                self.error = str(exc)
                if self.mode != "auto":
                    return
                self.stop_event.wait(2)

def open_camera():
    return CameraReader(
        LIVE_SOURCE, LIVE_TOKEN, LIVE_READ_MODE, LIVE_SNAPSHOT_FPS,
        CF_ACCESS_CLIENT_ID, CF_ACCESS_CLIENT_SECRET, LIVE_MJPEG_WAIT_S,
    )

def build_live_pipeline():
    if DETECTOR_DIR is None or detector is None:
        raise RuntimeError('Activa RUN_INFERENCE y ejecuta 4 antes de reconocer con cámara.')
    if not MODEL_IS_COUNTRY_FINETUNED:
        raise RuntimeError('La cámara requiere un candidato entrenado y validado para Ecuador.')
    live_confidence, threshold_source = validate_live_calibration(DETECTOR_DIR, LIVE_THRESHOLD_RECORD_PATH)
    detector_config = COUNTRY_CONFIG['detector']
    live_detector = PlateDetector(
        checkpoint=DETECTOR_DIR, model_id=MODEL_ID, imgsz=IMAGE_SIZE, conf=live_confidence,
        class_id=detector_config.get('class_id', 0), device='CPU', fallback_checkpoint=None,
    )
    live_reader = RuntimeSelectedSerialReader(model_name=OCR_MODEL_NAME)
    print('OCR Ecuador:', live_reader.runtime_name)
    return live_detector, live_reader, live_confidence, threshold_source


def recognize_live_frame(live_detector, live_reader, image_rgb):
    detections = live_detector.detect(image_rgb, max_detections=LIVE_MAX_CANDIDATES)
    best = None
    for detection in detections[:LIVE_MAX_CANDIDATES]:
        crop = crop_with_margin(image_rgb, detection.box, 0.06)
        if crop is None or crop.size == 0:
            continue
        raw, confidence_min, confidence_mean = live_reader.read(crop)
        raw_text = str(raw or "").strip().upper()
        plate, valid = normalize_plate(raw_text)
        formatted = format_plate(plate) if valid else ''
        candidate = {
            "plate_text": formatted or raw_text,
            "plate_raw": raw_text,
            "plate_key": formatted if valid and confidence_min >= OCR_THRESHOLD else '',
            "country": "ecuador",
            "ocr_confidence_min": float(confidence_min),
            "ocr_confidence_mean": float(confidence_mean),
            "det_confidence": float(detection.confidence),
            "box": detection.box,
        }
        if best is None or (
            candidate["ocr_confidence_min"], candidate["det_confidence"]
        ) > (
            best["ocr_confidence_min"], best["det_confidence"]
        ):
            best = candidate
        if candidate["plate_key"]:
            break
    return best

def annotate_live_frame(frame_bgr, recognition, camera_fps, process_fps):
    annotated = frame_bgr.copy()
    if recognition:
        box = recognition["box"]
        if box:
            x1, y1, x2, y2 = (int(round(value)) for value in box)
            height, width = annotated.shape[:2]
            x1, x2 = max(0, min(width - 1, x1)), max(0, min(width - 1, x2))
            y1, y2 = max(0, min(height - 1, y1)), max(0, min(height - 1, y2))
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 220, 0), 2)
            label = recognition["plate_text"] or "Plate detected; OCR unreadable"
            cv2.putText(
                annotated, str(label)[:32], (x1, max(24, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 0), 2, cv2.LINE_AA,
            )
    cv2.putText(
        annotated, f"Camera {camera_fps:.1f} fps | Processing {process_fps:.1f} fps",
        (12, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, cv2.LINE_AA,
    )
    return annotated

print("Live-camera configuration defined; no connection has been opened.")
print("Camera URL configured:", bool(LIVE_SOURCE))
print("Reader mode:", LIVE_READ_MODE)
print("Local live-session staging:", LIVE_OUTPUT_DIR)
print("Destino de la sesión (Drive solo si está activado):", LIVE_DRIVE_OUTPUT_DIR)
print("El modelo seleccionado está entrenado con Ecuador:", globals().get("MODEL_IS_COUNTRY_FINETUNED", False))
if not globals().get("MODEL_IS_COUNTRY_FINETUNED", False):
    print("La sesión de reconocimiento requiere un IR y una calibración Ecuador; el puente puede probarse aparte.")


### 7.3 Comprobar la conexión de la cámara

Activa `RUN_CAMERA_CONNECTION_CHECK=True` para conectarte cinco segundos sin cargar
el detector ni el OCR. Informa cuadros, FPS aproximados y resolución, y muestra el
último cuadro para revisar el ángulo. No guarda video ni mide exactitud de reconocimiento.
Ejecuta primero 7.2 para definir el lector y cargar los Secrets.


In [ ]:
LIVE_TEST_SECONDS = 5
RUN_CAMERA_CONNECTION_CHECK = False

if not RUN_CAMERA_CONNECTION_CHECK:
    print("Camera connection check is disabled. Set RUN_CAMERA_CONNECTION_CHECK = True to run the 5-second check.")
elif not LIVE_SOURCE:
    print("No camera URL is configured. Add CAMARA_URL and CAMARA_TOKEN to Colab Secrets, then rerun section 7.2.")
else:
    test_camera = open_camera()
    try:
        test_camera.start()
        start_time = time.monotonic()
        first_count = test_camera.received
        latest = None
        while time.monotonic() - start_time < LIVE_TEST_SECONDS:
            frame, count, _, _ = test_camera.snapshot()
            if frame is not None:
                latest = frame
            time.sleep(0.05)
        elapsed = max(time.monotonic() - start_time, 1e-6)
        final_count = test_camera.received
        if latest is None:
            detail = test_camera.error or "No frames arrived during the connection check."
            raise CameraUnavailable(detail)
        print({
            "frames_received": final_count - first_count,
            "approx_camera_fps": round((final_count - first_count) / elapsed, 2),
            "resolution": f"{latest.shape[1]} x {latest.shape[0]}",
            "reader_mode": LIVE_READ_MODE,
            "elapsed_seconds": round(elapsed, 2),
        })
        display(Image.fromarray(cv2.cvtColor(latest, cv2.COLOR_BGR2RGB)))
    except CameraUnavailable as exc:
        print("Camera connection check failed:", exc)
    finally:
        test_camera.stop()


### 7.4 Ejecutar la sesión y guardar sus salidas

Activa `LIVE_ENABLED=True` cuando estén listos modelo, calibración y cámara. Se procesa
el último cuadro disponible y se confirma una placa normalizada tras dos observaciones
aceptadas dentro de tres segundos; el período de supresión de repetidos es de quince
segundos. Los cuadros con OCR inferior a 0,94 no aportan confirmaciones.

La sesión se limita por `LIVE_DURATION_S` (120 segundos inicialmente), falta de cuadros
recientes o interrupción. Guarda `readings.csv` y `summary.json`; con `LIVE_SAVE_VIDEO=True`
añade un MP4 anotado. El CSV incluye país, texto OCR original, placa normalizada y confianzas.
No usa la lista autorizada ni produce decisiones de acceso.

Las salidas quedan en `WORK_DIR/live_camera/ecuador_live_…/`. Si activaste Drive en 0.2,
se copian al terminar a `SIRAP_USA_ECUADOR/private/ecuador/camera_sessions/`. No se publican
los cuadros o las placas en GitHub. El reconocimiento en vivo no ejecuta consultas SRI/propietario.


In [ ]:
from collections import defaultdict, deque
from datetime import datetime, timezone
import csv
import json

LIVE_ENABLED = False

def run_live_session():
    live_detector, live_reader, live_confidence, threshold_source = build_live_pipeline()
    reader = open_camera()
    session_id = datetime.now(timezone.utc).strftime("ecuador_live_%Y%m%dT%H%M%S%fZ")
    session_dir = LIVE_OUTPUT_DIR / session_id
    drive_session_dir = LIVE_DRIVE_OUTPUT_DIR / session_id
    session_dir.mkdir(parents=True, exist_ok=False)
    readings_path = session_dir / "readings.csv"
    summary_path = session_dir / "summary.json"
    video_path = session_dir / "annotated.mp4"

    observations = defaultdict(lambda: deque(maxlen=LIVE_CONFIRMATIONS))
    last_logged = {}
    rows = []
    latencies_ms = []
    writer = None
    display_handle = None
    started = time.monotonic()
    last_processed_id = -1
    last_processed_at = 0.0
    last_display_at = 0.0
    processed = 0

    if LIVE_SAVE_VIDEO:
        print("Annotated video recording is enabled.")

    reader.start()
    try:
        initial_deadline = time.monotonic() + LIVE_START_TIMEOUT_S
        while reader.received == 0 and time.monotonic() < initial_deadline:
            if reader.error and LIVE_READ_MODE != "auto":
                raise CameraUnavailable(reader.error)
            time.sleep(0.1)
        if reader.received == 0:
            raise CameraUnavailable(reader.error or "No camera frame arrived before the startup timeout.")

        while True:
            now = time.monotonic()
            if LIVE_DURATION_S is not None and now - started >= LIVE_DURATION_S:
                break
            frame_bgr, frame_id, last_frame_at, captured_epoch = reader.snapshot()
            if frame_bgr is None:
                time.sleep(0.02)
                continue
            if last_frame_at is not None and now - last_frame_at > LIVE_STALE_S:
                raise CameraUnavailable("The camera stream stopped sending frames.")
            if frame_id == last_processed_id:
                time.sleep(0.01)
                continue
            if now - last_processed_at < 1.0 / max(LIVE_MAX_PROCESS_FPS, 0.1):
                time.sleep(0.01)
                continue

            last_processed_id = frame_id
            last_processed_at = now
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            inference_started = time.perf_counter()
            recognition = recognize_live_frame(live_detector, live_reader, frame_rgb)
            inference_ms = (time.perf_counter() - inference_started) * 1000
            latencies_ms.append(inference_ms)
            processed += 1

            elapsed_now = max(now - started, 1e-6)
            camera_fps = reader.received / elapsed_now
            process_fps = processed / elapsed_now
            annotated = annotate_live_frame(frame_bgr, recognition, camera_fps, process_fps)

            raw_text = recognition["plate_text"] if recognition else ""
            plate_key = recognition["plate_key"] if recognition else ""
            if plate_key:
                history = observations[plate_key]
                while history and now - history[0] > LIVE_CONFIRM_WINDOW_S:
                    history.popleft()
                history.append(now)
                last_time = last_logged.get(plate_key, 0.0)
                if len(history) >= LIVE_CONFIRMATIONS and now - last_time >= LIVE_DUPLICATE_COOLDOWN_S:
                    rows.append({
                        "timestamp_utc": datetime.fromtimestamp(
                            captured_epoch or time.time(), tz=timezone.utc
                        ).isoformat(),
                        "country": "ecuador",
                        "plate_raw": recognition["plate_raw"],
                        "plate_text": raw_text,
                        "plate_key": plate_key,
                        "detector_confidence": recognition["det_confidence"],
                        "ocr_confidence_min": recognition["ocr_confidence_min"],
                        "ocr_confidence_mean": recognition["ocr_confidence_mean"],
                        "box_xyxy": json.dumps(recognition["box"]),
                        "confirmation_count": len(history),
                        "model": "RT-DETRv2-S",
                        "runtime": "OpenVINO Runtime",
                    })
                    last_logged[plate_key] = now

            if LIVE_SAVE_VIDEO:
                height, width = annotated.shape[:2]
                if writer is None:
                    writer = cv2.VideoWriter(
                        str(video_path), cv2.VideoWriter_fourcc(*"mp4v"),
                        max(1.0, min(float(LIVE_MAX_PROCESS_FPS), 10.0)), (width, height),
                    )
                    if not writer.isOpened():
                        writer.release()
                        writer = None
                        raise RuntimeError("Could not create the annotated MP4 output.")
                writer.write(annotated)

            if now - last_display_at >= 1.0 / max(LIVE_DISPLAY_FPS, 0.1):
                preview = Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
                if display_handle is None:
                    display_handle = display(preview, display_id=True)
                else:
                    display_handle.update(preview)
                last_display_at = now
    except KeyboardInterrupt:
        print("Live session interrupted by the user; saving the readings collected so far.")
    finally:
        reader.stop()
        if writer is not None:
            writer.release()

    elapsed = max(time.monotonic() - started, 1e-6)
    readings_df = pd.DataFrame(
        rows,
        columns=[
            "timestamp_utc", "country", "plate_raw", "plate_text", "plate_key", "detector_confidence",
            "ocr_confidence_min", "ocr_confidence_mean", "box_xyxy",
            "confirmation_count", "model", "runtime",
        ],
    )
    readings_df.to_csv(readings_path, index=False)
    summary = {
        "session_id": session_id,
        "country": "ecuador",
        "model": "RT-DETRv2-S",
        "runtime": "OpenVINO Runtime",
        "country_finetuned_checkpoint": bool(globals().get("MODEL_IS_COUNTRY_FINETUNED", False)),
        "confidence_threshold": live_confidence,
        "ocr_threshold": OCR_THRESHOLD,
        "threshold_source": threshold_source,
        "duration_seconds": round(elapsed, 2),
        "frames_received": int(reader.received),
        "frames_processed": int(processed),
        "camera_fps": round(reader.received / elapsed, 3),
        "processing_fps": round(processed / elapsed, 3),
        "mean_pipeline_latency_ms": (
            round(float(np.mean(latencies_ms)), 2) if latencies_ms else None
        ),
        "confirmed_readings": len(rows),
        "readings_csv": str(drive_session_dir / "readings.csv"),
        "annotated_video": (
            str(drive_session_dir / "annotated.mp4")
            if LIVE_SAVE_VIDEO and video_path.is_file() else None
        ),
        "authorization_decisions": False,
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    persist_tree_to_drive(session_dir, drive_session_dir)
    print("Live session summary:")
    display(pd.Series(summary, name=session_id).to_frame())
    print("Confirmed readings:")
    safe_readings_preview = readings_df.drop(columns=["plate_raw", "plate_text", "plate_key"], errors="ignore")
    display(safe_readings_preview)
    print("Local session staging:", session_dir)
    print("Destino de archivos (Drive si está activado):", drive_session_dir)
    if not globals().get("MODEL_IS_COUNTRY_FINETUNED", False):
        print("Reminder: this run used the shared technical fallback, no evidencia de rendimiento Ecuador.")
    return summary

live_summary = None
if not LIVE_ENABLED:
    print("Live processing is disabled. Set LIVE_ENABLED = True to run a deliberate camera session.")
elif not LIVE_SOURCE:
    print("Live processing skipped: add CAMARA_URL and CAMARA_TOKEN to Colab Secrets first.")
else:
    try:
        live_summary = run_live_session()
    except CameraUnavailable as exc:
        print("Live camera session stopped:", exc)


### 7.5 Generar el ejecutor local para el PC de la cámara

Este segundo modo mantiene las imágenes en el PC: ejecuta RT-DETRv2-S con OpenVINO y
OCR ONNX, aplica patrones Ecuador, confirma lecturas y guarda un CSV local. Genera
`WORK_DIR/live_camera/edge_inference_ecuador.py`, con copia a Drive solo si se activó 0.2.

En el PC necesitas un clon de SIRAP_USA_ECUADOR, las dependencias del backend y los cuatro
archivos del IR Ecuador bajo `models/ecuador/detector/rtdetrv2_s/openvino/`, o una ruta
pasada con `--checkpoint`. Instala `fast-plate-ocr[onnx-openvino]==1.1.0` en Intel o
`fast-plate-ocr[onnx]==1.1.0` en otros procesadores, además de OpenVINO y las dependencias
del proyecto. El OCR global descarga sus pesos en el primer uso.

Copia también el informe de calibración Ecuador de 3.2 y ejecuta, por ejemplo:

```bash
python edge_inference_ecuador.py --source 0 --calibration results/ecuador/metrics/rtdetr_ecuador_threshold_calibration.json --duration 120
```

El script valida país y hashes antes de abrir la cámara. `--threshold`, si se proporciona,
debe coincidir con el umbral del informe; `--ocr-threshold` usa 0,94 inicialmente. El CSV
se guarda bajo `~/.sirap/live/ecuador/`, o en `SIRAP_EDGE_OUTPUT_DIR` si se configura.
No guarda ni transmite video o recortes. El envío de eventos permanece desactivado;
el modo opcional `--upload-events` necesita una API implementada para `/api/edge/events`
y Secrets de entorno, y envía `country="ecuador"`. No se usa un endpoint de operaciones de barrera.


In [ ]:
from pathlib import Path

EDGE_RUNNER_PATH = WORK_DIR / "live_camera/edge_inference_ecuador.py"
EDGE_RUNNER_PATH.parent.mkdir(parents=True, exist_ok=True)

EDGE_RUNNER_SCRIPT = r'''
import argparse
import hashlib
import math
import csv
import json
import os
import sys
import threading
import time
import uuid
from collections import defaultdict, deque
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

def locate_repository():
    hints = []
    if os.environ.get("SIRAP_REPO_ROOT"):
        hints.append(Path(os.environ["SIRAP_REPO_ROOT"]).expanduser())
    hints.append(Path.cwd())
    hints.extend(Path(__file__).resolve().parents)
    for candidate in hints:
        if (candidate / "src" / "model.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this script from inside a SIRAP_USA_ECUADOR clone or set SIRAP_REPO_ROOT."
    )

REPO_ROOT = locate_repository()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import cv2
import numpy as np
import requests
import yaml
from src.data_processing import clean_text, crop_with_margin, normalize_plate
from src.model import PlateDetector

class RuntimeSelectedSerialReader:
    """ONNX OCR with OpenVINO EP on Intel CPU, standard ORT CPU on other CPUs."""

    def __init__(self, model_name="cct-s-v2-global-model"):
        import platform
        import re
        import onnxruntime as ort
        from fast_plate_ocr import LicensePlateRecognizer

        details = f"{platform.processor()} {platform.uname().processor}"
        cpuinfo = Path("/proc/cpuinfo")
        if cpuinfo.is_file():
            try:
                details += " " + cpuinfo.read_text(errors="ignore")
            except OSError:
                pass
        is_intel = bool(re.search(
            r"genuineintel|intel\(r\)|\bintel\b", details, re.I
        )) and not bool(re.search(r"authenticamd|\bamd\b", details, re.I))

        available = ort.get_available_providers()
        if is_intel:
            if "OpenVINOExecutionProvider" not in available:
                raise RuntimeError(
                    "Intel CPU detected, but OpenVINOExecutionProvider is missing. "
                    "Install fast-plate-ocr[onnx-openvino]."
                )
            providers = ["OpenVINOExecutionProvider"]
            if "CPUExecutionProvider" in available:
                providers.append("CPUExecutionProvider")
        else:
            if "CPUExecutionProvider" not in available:
                raise RuntimeError("Install fast-plate-ocr[onnx] for ONNX Runtime CPU inference.")
            providers = ["CPUExecutionProvider"]

        self.model = LicensePlateRecognizer(model_name, providers=providers)
        self.providers = self.model.model.get_providers()
        if is_intel and (
            not self.providers or self.providers[0] != "OpenVINOExecutionProvider"
        ):
            raise RuntimeError(
                f"OpenVINOExecutionProvider is not first in the Intel session: {self.providers}"
            )
        if not is_intel and (
            not self.providers or self.providers[0] != "CPUExecutionProvider"
        ):
            raise RuntimeError(f"ONNX Runtime CPU provider not active: {self.providers}")
        self.runtime_name = (
            "ONNX Runtime + OpenVINOExecutionProvider"
            if is_intel else "ONNX Runtime + CPUExecutionProvider"
        )

    def read(self, crop):
        prediction = self.model.run(
            np.ascontiguousarray(crop),
            return_confidence=True,
        )[0]
        text = clean_text(prediction.plate)
        probabilities = np.asarray(
            prediction.char_probs, dtype=np.float32
        ).ravel()[:max(len(text), 1)]
        if not probabilities.size:
            return text, 0.0, 0.0
        return text, float(probabilities.min()), float(probabilities.mean())

def require_ecuador_ir(directory):
    """Comprueba los cuatro archivos del bundle y su identificación Ecuador."""
    directory = Path(directory)
    required = ('openvino_model.xml', 'openvino_model.bin', 'sirap_openvino.json', 'sirap_training.json')
    missing = [name for name in required if not (directory / name).is_file()]
    if missing:
        raise FileNotFoundError(f'IR Ecuador incompleto en {directory}: {missing}')
    manifest = json.loads((directory / 'sirap_training.json').read_text(encoding='utf-8'))
    if manifest.get('country') != 'ecuador':
        raise ValueError('El manifiesto de entrenamiento no identifica un candidato Ecuador.')
    return directory

def validate_live_calibration(directory, record_path):
    """Vincula el umbral validado de Ecuador a los bytes exactos del XML y BIN."""
    directory = require_ecuador_ir(directory)
    record_path = Path(record_path)
    if not record_path.is_file():
        raise FileNotFoundError(f'Falta la calibración Ecuador sobre validación: {record_path}. Ver 3.2.')
    record = json.loads(record_path.read_text(encoding='utf-8'))
    threshold = record.get('threshold')
    if (record.get('country') != 'ecuador' or record.get('usable_for_inference') is not True
            or isinstance(threshold, bool) or not isinstance(threshold, (int, float))
            or not math.isfinite(threshold) or not 0 <= threshold <= 1):
        raise ValueError('La calibración Ecuador no habilita inferencia o no tiene un umbral válido.')
    expected = record.get('checkpoint_sha256', {})
    for name in ('openvino_model.xml', 'openvino_model.bin'):
        actual = hashlib.sha256((directory / name).read_bytes()).hexdigest()
        if expected.get(name) != actual:
            raise ValueError(f'El informe de calibración no corresponde al IR seleccionado: {name}.')
    return float(threshold), 'calibración Ecuador sobre validación'

def parse_source(value):
    return int(value) if str(value).isdigit() else value

class LatestCamera:
    def __init__(self, source):
        self.source = parse_source(source)
        self.lock = threading.Lock()
        self.stop_event = threading.Event()
        self.frame = None
        self.frame_id = 0
        self.received_at = None
        self.error = ""

    def start(self):
        self.thread = threading.Thread(target=self._capture, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        self.thread.join(timeout=3)

    def latest(self):
        with self.lock:
            if self.frame is None:
                return None, self.frame_id, None
            return self.frame.copy(), self.frame_id, self.received_at

    def _capture(self):
        while not self.stop_event.is_set():
            camera = cv2.VideoCapture(self.source)
            camera.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            if not camera.isOpened():
                self.error = "OpenCV could not open the configured camera."
                camera.release()
                self.stop_event.wait(2)
                continue
            while not self.stop_event.is_set():
                ok, frame = camera.read()
                if not ok:
                    self.error = "Camera frame read failed; reconnecting."
                    break
                with self.lock:
                    self.frame = frame
                    self.frame_id += 1
                    self.received_at = time.time()
                    self.error = ""
            camera.release()
            if not self.stop_event.is_set():
                self.stop_event.wait(1)

def recognize(detector, reader, image_rgb, max_candidates=5):
    detections = detector.detect(image_rgb, max_detections=max_candidates)
    best = None
    for detection in detections[:max_candidates]:
        crop = crop_with_margin(image_rgb, detection.box, 0.06)
        if crop is None or crop.size == 0:
            continue
        raw, confidence_min, confidence_mean = reader.read(crop)
        raw_text = str(raw or "").strip().upper()
        formatted = normalize_plate(raw_text, ((3, 4), (3, 3))) or ''
        candidate = {
            "plate_raw": raw_text,
            "plate_text": formatted or raw_text,
            "plate_key": formatted,
            "detector_confidence": float(detection.confidence),
            "ocr_confidence_min": float(confidence_min),
            "ocr_confidence_mean": float(confidence_mean),
            "box": tuple(float(value) for value in detection.box),
        }
        if best is None or (
            candidate["ocr_confidence_min"], candidate["detector_confidence"]
        ) > (
            best["ocr_confidence_min"], best["detector_confidence"]
        ):
            best = candidate
        if candidate["plate_key"]:
            break
    return best

def send_event_if_enabled(event, enabled):
    if not enabled:
        return "local_only"
    endpoint = os.environ.get("SIRAP_EDGE_EVENTS_URL", "").strip()
    token = os.environ.get("SIRAP_EDGE_EVENTS_TOKEN", "").strip()
    if not endpoint or not token:
        raise RuntimeError(
            "Event upload needs SIRAP_EDGE_EVENTS_URL and SIRAP_EDGE_EVENTS_TOKEN."
        )
    parsed = urlsplit(endpoint)
    if parsed.scheme != "https":
        raise ValueError("The edge event endpoint must use HTTPS.")
    if parsed.path.rstrip("/") != "/api/edge/events":
        raise ValueError(
            "For safety, uploads are restricted to the dedicated /api/edge/events route."
        )
    response = requests.post(
        endpoint,
        json=event,
        headers={"Authorization": "Bearer " + token},
        timeout=(5, 10),
    )
    response.raise_for_status()
    return "sent"

def main():
    ecuador_config = yaml.safe_load(
        (REPO_ROOT / "config" / "countries" / "ecuador.yaml").read_text(encoding="utf-8")
    )
    detector_config = ecuador_config["detector"]
    checkpoint = REPO_ROOT / "models/ecuador/detector/rtdetrv2_s/openvino"
    parser = argparse.ArgumentParser(
        description="Run Ecuador plate detection locally beside the camera."
    )
    parser.add_argument(
        "--source", default=os.environ.get("SIRAP_CAMERA_SOURCE", "0"),
        help="Local camera index, local video path, or source URL.",
    )
    parser.add_argument("--checkpoint", default=str(checkpoint))
    parser.add_argument("--calibration", required=True,
                        help="Ecuador validation report with threshold and XML/BIN hashes.")
    parser.add_argument("--threshold", type=float, default=None,
                        help="Optional check; must match the validated threshold.")
    parser.add_argument("--ocr-threshold", type=float, default=0.94)
    parser.add_argument("--max-fps", type=float, default=5)
    parser.add_argument("--duration", type=float, default=0,
                        help="Seconds to run; 0 means run until interrupted.")
    parser.add_argument("--min-confirmations", type=int, default=2)
    parser.add_argument("--confirm-window", type=float, default=3)
    parser.add_argument("--duplicate-cooldown", type=float, default=15)
    parser.add_argument("--camera-id", default=os.environ.get("SIRAP_CAMERA_ID", "ecuador-edge-01"))
    parser.add_argument("--preview", action="store_true")
    parser.add_argument(
        "--upload-events", action="store_true",
        help="Send confirmed text events to a dedicated HTTPS Ecuador intake endpoint.",
    )
    args = parser.parse_args()
    checkpoint = Path(args.checkpoint).expanduser().resolve()
    calibrated_threshold, _ = validate_live_calibration(checkpoint, Path(args.calibration).expanduser())
    if args.threshold is not None and not math.isclose(args.threshold, calibrated_threshold, abs_tol=1e-9):
        raise ValueError('--threshold no coincide con el informe de calibración Ecuador.')
    args.threshold = calibrated_threshold
    if not math.isfinite(args.ocr_threshold) or not 0 <= args.ocr_threshold <= 1:
        raise ValueError('--ocr-threshold debe estar entre 0 y 1.')

    if not 0 <= args.threshold <= 1:
        raise ValueError("--threshold must be between 0 and 1.")
    if not checkpoint.is_dir():
        raise FileNotFoundError(
            "The Ecuador fine-tuned OpenVINO folder is missing: " + str(checkpoint)
        )
    if not any(checkpoint.glob("*.xml")) or not any(checkpoint.glob("*.bin")):
        raise FileNotFoundError(
            "Copy both OpenVINO IR files (.xml and .bin) into " + str(checkpoint)
        )

    detector = PlateDetector(
        checkpoint=checkpoint,
        model_id=detector_config["model_id"],
        imgsz=int(detector_config.get("imgsz_inferencia", 640)),
        conf=args.threshold,
        class_id=int(detector_config.get("class_id", 0)),
        device="CPU",
        fallback_checkpoint=None,
    )
    reader = RuntimeSelectedSerialReader()
    print("Registration-serial OCR backend:", reader.runtime_name)
    camera = LatestCamera(args.source).start()
    output_dir = Path(os.environ.get("SIRAP_EDGE_OUTPUT_DIR", str(Path.home() / ".sirap" / "live" / "ecuador"))).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)
    session_id = datetime.now(timezone.utc).strftime("ecuador_edge_%Y%m%dT%H%M%SZ")
    output_path = output_dir / (session_id + "_readings.csv")
    fieldnames = [
        "event_id", "timestamp_utc", "country", "camera_id", "plate_raw", "plate_text",
        "plate_key", "detector_confidence", "ocr_confidence_min",
        "ocr_confidence_mean", "model", "runtime", "upload_status",
    ]
    history = defaultdict(lambda: deque(maxlen=max(1, args.min_confirmations)))
    last_sent = {}
    last_frame_id = -1
    last_process_at = 0.0
    started = time.monotonic()
    processed = 0
    rows_written = 0
    last_frame_at = None
    source = str(args.source)
    print("Ecuador model:", detector.model_path)
    print("Threshold:", args.threshold)
    print("Camera source:", source if source.isdigit() else "(configured source)")
    print("Event upload:", "enabled" if args.upload_events else "disabled; local CSV only")
    print("Calibración Ecuador verificada para este IR.")

    try:
        with output_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            while True:
                now = time.monotonic()
                if args.duration and now - started >= args.duration:
                    break
                frame_bgr, frame_id, received_at = camera.latest()
                if frame_bgr is None:
                    if camera.error:
                        print(camera.error)
                    if now - started > 30:
                        raise RuntimeError("No camera frame arrived within 30 seconds.")
                    time.sleep(0.05)
                    continue
                if frame_id == last_frame_id:
                    if last_frame_at and now - last_frame_at > 20:
                        raise RuntimeError("The camera stream has stopped.")
                    time.sleep(0.01)
                    continue
                if now - last_process_at < 1.0 / max(args.max_fps, 0.1):
                    time.sleep(0.01)
                    continue

                last_frame_id = frame_id
                last_process_at = now
                last_frame_at = now
                image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                result = recognize(detector, reader, image_rgb)
                processed += 1

                if result and result["plate_key"] and result["ocr_confidence_min"] >= args.ocr_threshold:
                    key = result["plate_key"]
                    samples = history[key]
                    while samples and now - samples[0] > args.confirm_window:
                        samples.popleft()
                    samples.append(now)
                    if (
                        len(samples) >= max(1, args.min_confirmations)
                        and now - last_sent.get(key, 0) >= args.duplicate_cooldown
                    ):
                        event = {
                            "event_id": str(uuid.uuid4()),
                            "timestamp_utc": datetime.fromtimestamp(
                                received_at or time.time(), tz=timezone.utc
                            ).isoformat(),
                            "country": "ecuador",
                            "camera_id": args.camera_id,
                            "plate_raw": result["plate_raw"],
                            "plate_text": result["plate_text"],
                            "plate_key": key,
                            "detector_confidence": result["detector_confidence"],
                            "ocr_confidence_min": result["ocr_confidence_min"],
                            "ocr_confidence_mean": result["ocr_confidence_mean"],
                            "model": "RT-DETRv2-S",
                            "runtime": "OpenVINO Runtime",
                        }
                        try:
                            upload_status = send_event_if_enabled(event, args.upload_events)
                        except Exception as exc:
                            upload_status = "failed:" + type(exc).__name__
                            print("Event upload failed; the local row is still saved:", exc)
                        writer.writerow({**event, "upload_status": upload_status})
                        handle.flush()
                        rows_written += 1
                        last_sent[key] = now
                        print("Confirmed plate text:", result["plate_text"], "|", upload_status)

                if args.preview:
                    shown = frame_bgr.copy()
                    if result and result.get("box"):
                        x1, y1, x2, y2 = (int(round(value)) for value in result["box"])
                        cv2.rectangle(shown, (x1, y1), (x2, y2), (0, 220, 0), 2)
                        cv2.putText(shown, result["plate_text"] or "Plate detected", (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 0), 2, cv2.LINE_AA)
                    cv2.imshow("SIRAP Ecuador edge inference", shown)
                    if cv2.waitKey(1) & 0xFF == ord("q"):
                        break
    except KeyboardInterrupt:
        print("Edge session interrupted; the local CSV has been saved.")
    finally:
        camera.stop()
        if args.preview:
            cv2.destroyAllWindows()

    elapsed = max(time.monotonic() - started, 1e-6)
    print(json.dumps({
        "session_id": session_id,
        "frames_received": camera.frame_id,
        "frames_processed": processed,
        "confirmed_readings": rows_written,
        "elapsed_seconds": round(elapsed, 2),
        "processing_fps": round(processed / elapsed, 3),
        "local_csv": str(output_path),
        "video_uploaded": False,
    }, indent=2))

if __name__ == "__main__":
    main()
'''

EDGE_RUNNER_PATH.write_text(EDGE_RUNNER_SCRIPT, encoding="utf-8")
edge_persistent_path = persist_file_to_drive(
    EDGE_RUNNER_PATH,
    DRIVE_SCRIPTS_DIR / "live_camera" / "edge_inference_ecuador.py",
)
print("On-site edge runner created:", EDGE_RUNNER_PATH)
print("Archivo reutilizable:", edge_persistent_path)
print("Download it from Colab Files and place it in the local project clone before running it.")


## 8. Medir la latencia de inferencia en CPU

Integrada de la sección 18 USA. Activa `RUN_LATENCY_TEST` cuando estén listos el COCO
Ecuador de 1.2 y el IR Ecuador cargado en 4. Usa hasta 200 imágenes de **validación**,
calienta con las primeras diez (o menos si el conjunto es pequeño) y mide las restantes.
Necesita al menos dos imágenes para separar calentamiento y medición.

Registra mediana, p95, p99 y FPS aproximados desde la mediana, además de CPU, tamaño de
entrada, checkpoint y umbral. Mide solo el detector: excluye lectura de imagen, OCR,
transferencia de red y lógica de acceso. No usa el test reservado ni mide latencia total.
El resumen agregado queda en `results/ecuador/metrics/openvino_ecuador_latency.json`;
su publicación requiere activar `PUBLISH_SMALL_METRICS_TO_GITHUB`.


In [ ]:
from src.data_processing import load_image

RUN_LATENCY_TEST = False
if RUN_LATENCY_TEST:
    if not DATA_READY or detector is None or not MODEL_IS_COUNTRY_FINETUNED:
        raise RuntimeError("A COCO validation split and the Ecuador fine-tuned IR are required.")
    validation_dataset = SPLITS["val"]["dataset"]
    samples = validation_dataset.images[:min(200, len(validation_dataset.images))]
    if len(samples) < 2:
        raise ValueError('Se necesitan al menos dos imágenes de validación para calentar y medir.')
    warmup_count = min(10, len(samples) - 1)
    for item in samples[:warmup_count]:
        image = load_image(validation_dataset._image_path(item["file_name"]).read_bytes())
        detector.detect(image, max_detections=100)
    latencies_ms = []
    for item in samples[warmup_count:]:
        image = load_image(validation_dataset._image_path(item["file_name"]).read_bytes())
        start = time.perf_counter()
        detector.detect(image, max_detections=100)
        latencies_ms.append((time.perf_counter() - start) * 1000)
    latency_summary = {
        "n": len(latencies_ms),
        "device": "CPU",
        "country": COUNTRY,
        "warmup_n": warmup_count,
        "cpu": platform.processor() or platform.machine(),
        "input_size": IMAGE_SIZE,
        "checkpoint": str(DETECTOR_DIR),
        "detector_threshold": float(detector.conf),
        "median_ms": float(np.median(latencies_ms)) if latencies_ms else None,
        "p95_ms": float(np.percentile(latencies_ms, 95)) if latencies_ms else None,
        "p99_ms": float(np.percentile(latencies_ms, 99)) if latencies_ms else None,
        "fps_from_median": 1000 / float(np.median(latencies_ms)) if latencies_ms and float(np.median(latencies_ms)) > 0 else None,
    }
    latency_metrics_path = METRICS_DIR / "openvino_ecuador_latency.json"
    latency_metrics_path.write_text(
        json.dumps(latency_summary, indent=2), encoding="utf-8"
    )
    publish_small_metrics(
        [latency_metrics_path],
        "Publicar resumen de latencia OpenVINO Ecuador",
    )
    print(latency_summary)
else:
    print("Latency measurement is disabled.")


## 9. Publicar opcionalmente el IR OpenVINO revisado

Integrada de la sección 25 USA. `PUBLISH_IR_TO_SPACE=False` inicialmente. Al activarla,
comprueba el bundle Ecuador y copia solo el IR a `models/ecuador/detector/rtdetrv2_s/openvino/`
del clon, preservando las demás carpetas. La función existente `publish_detector_ir`
publica esa ruta con `GITHUB_TOKEN` de Colab; la automatización de GitHub puede después
sincronizarla al Space. No se incluyen datasets, checkpoint Transformers ni CSV de placas.

Revisa primero validación y evaluación independiente Ecuador. Cargar el modelo y medir
latencia no demuestra exactitud. Los modelos compartidos o iniciales usados como prueba
técnica no se publican como candidatos Ecuador. Esta sección tampoco cambia por sí sola
el checkpoint configurado en la aplicación; comprueba esa configuración antes de activarlo.


In [ ]:
PUBLISH_IR_TO_SPACE = False
if PUBLISH_IR_TO_SPACE:
    from scripts.notebook_io import publish_detector_ir
    if not MODEL_IS_COUNTRY_FINETUNED or DETECTOR_DIR is None:
        raise RuntimeError('Carga en 4 el IR Ecuador entrenado y revisado antes de publicarlo.')
    source_ir = require_ecuador_ir(DETECTOR_DIR)
    staged = subprocess.run(['git', '-C', str(REPO_ROOT), 'diff', '--cached', '--name-only'],
                            check=True, capture_output=True, text=True).stdout.strip()
    if staged:
        raise RuntimeError('Hay archivos previamente preparados en Git. Revísalos antes de publicar el IR.')
    target_ir = REPO_MODEL_OUTPUT / 'openvino'
    if source_ir.resolve() != target_ir.resolve():
        target_ir.parent.mkdir(parents=True, exist_ok=True)
        if target_ir.exists():
            shutil.rmtree(target_ir)  # Solo reemplaza el bundle IR; conserva el resto de modelos/ecuador.
        shutil.copytree(source_ir, target_ir)
    require_ecuador_ir(target_ir)
    print('IR Ecuador preparado en:', target_ir)
    print(publish_detector_ir(REPO_ROOT, COUNTRY))
else:
    print('Publicación desactivada. No se subió ningún modelo a GitHub ni al Space.')
